In [1]:
# ============================================================================
# NOTEBOOK CODE CELL 1
# ============================================================================
# ============================================================
# GOVERNANCE + STRATEGY SECTION GENERATORS
# Role-based Azure OpenAI REST endpoints
# Writer: GPT-5.1 | Judge: GPT-5.2 (LLM-only evaluation) | Reviser: GPT-4.1
# ============================================================

import os
import json
import re
import urllib.request
import urllib.error
import time
import random
from typing import TypedDict, Literal
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from langgraph.graph import StateGraph, START, END

# ── ENV LOADING ──────────────────────────────────────────────
env_path = find_dotenv()
if env_path:
    load_dotenv(env_path, override=True)
    print(f"Loaded .env from: {env_path}")
else:
    load_dotenv(override=True)
    print("No .env found by find_dotenv(); using existing environment variables.")

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")

_shared_key_fallback = (
    AZURE_OPENAI_API_KEY
    or os.getenv("AZURE_OPENAI_WRITER_API_KEY")
    or os.getenv("AZURE_OPENAI_JUDGE_API_KEY")
    or os.getenv("AZURE_OPENAI_REVISER_API_KEY")
)

AZURE_OPENAI_WRITER_API_KEY = os.getenv("AZURE_OPENAI_WRITER_API_KEY") or _shared_key_fallback
AZURE_OPENAI_JUDGE_API_KEY  = os.getenv("AZURE_OPENAI_JUDGE_API_KEY")  or _shared_key_fallback
AZURE_OPENAI_REVISER_API_KEY = os.getenv("AZURE_OPENAI_REVISER_API_KEY") or _shared_key_fallback

AZURE_OPENAI_WRITER_URL  = os.getenv("AZURE_OPENAI_WRITER_URL")  or os.getenv("AZURE_OPENAI_CHAT_URL")
AZURE_OPENAI_JUDGE_URL   = os.getenv("AZURE_OPENAI_JUDGE_URL")
AZURE_OPENAI_REVISER_URL = os.getenv("AZURE_OPENAI_REVISER_URL")


def _clean_url(value):
    if not value:
        return None
    return value.strip().strip('"').strip("'")


AZURE_OPENAI_WRITER_URL  = _clean_url(AZURE_OPENAI_WRITER_URL)
AZURE_OPENAI_JUDGE_URL   = _clean_url(AZURE_OPENAI_JUDGE_URL)
AZURE_OPENAI_REVISER_URL = _clean_url(AZURE_OPENAI_REVISER_URL)


def validate_role_config() -> None:
    required = {
        "AZURE_OPENAI_WRITER_API_KEY": AZURE_OPENAI_WRITER_API_KEY,
        "AZURE_OPENAI_JUDGE_API_KEY":  AZURE_OPENAI_JUDGE_API_KEY,
        "AZURE_OPENAI_REVISER_API_KEY": AZURE_OPENAI_REVISER_API_KEY,
        "AZURE_OPENAI_WRITER_URL":     AZURE_OPENAI_WRITER_URL,
        "AZURE_OPENAI_JUDGE_URL":      AZURE_OPENAI_JUDGE_URL,
        "AZURE_OPENAI_REVISER_URL":    AZURE_OPENAI_REVISER_URL,
    }
    missing = [name for name, value in required.items() if not value]
    if missing:
        loaded_flags = {
            "shared_key_loaded":  bool(AZURE_OPENAI_API_KEY),
            "writer_key_loaded":  bool(AZURE_OPENAI_WRITER_API_KEY),
            "judge_key_loaded":   bool(AZURE_OPENAI_JUDGE_API_KEY),
            "reviser_key_loaded": bool(AZURE_OPENAI_REVISER_API_KEY),
            "writer_url_loaded":  bool(AZURE_OPENAI_WRITER_URL),
            "judge_url_loaded":   bool(AZURE_OPENAI_JUDGE_URL),
            "reviser_url_loaded": bool(AZURE_OPENAI_REVISER_URL),
        }
        raise ValueError(
            "Missing role-based Azure configuration values: "
            + ", ".join(missing)
            + "\n\nLoaded configuration flags (keys are never printed):\n"
            + json.dumps(loaded_flags, indent=2)
        )
    for name, url in {
        "AZURE_OPENAI_WRITER_URL":  AZURE_OPENAI_WRITER_URL,
        "AZURE_OPENAI_JUDGE_URL":   AZURE_OPENAI_JUDGE_URL,
        "AZURE_OPENAI_REVISER_URL": AZURE_OPENAI_REVISER_URL,
    }.items():
        if not url.startswith("https://"):
            raise ValueError(f"{name} must be a full HTTPS Azure deployment URL: {url!r}")


validate_role_config()
print("Role-based Azure OpenAI configuration loaded")


def _azure_chat_completion(
    *,
    url: str,
    api_key: str,
    messages: list,
    max_output_tokens: int,
    json_mode: bool = False,
    temperature=None,
    use_max_completion_tokens: bool = False,
    timeout: int = 240,
    request_label: str = "LLM",
    max_attempts: int = 4,
) -> dict:
    preferred_field = "max_completion_tokens" if use_max_completion_tokens else "max_tokens"
    token_fields = [preferred_field]
    if preferred_field == "max_completion_tokens":
        token_fields.append("max_tokens")

    last_error = None
    for token_field in token_fields:
        payload = {"messages": messages, token_field: max_output_tokens}
        if temperature is not None:
            payload["temperature"] = temperature
        if json_mode:
            payload["response_format"] = {"type": "json_object"}

        for attempt in range(1, max_attempts + 1):
            req = urllib.request.Request(
                url,
                data=json.dumps(payload).encode("utf-8"),
                headers={"Content-Type": "application/json", "api-key": api_key},
                method="POST",
            )
            try:
                with urllib.request.urlopen(req, timeout=timeout) as resp:
                    return json.loads(resp.read().decode("utf-8"))
            except urllib.error.HTTPError as exc:
                body = exc.read().decode(errors="replace")
                last_error = RuntimeError(
                    f"{request_label} HTTP error {exc.code}.\nEndpoint: {url}\n"
                    f"Token field: {token_field}\nResponse: {body[:3000]}"
                )
                transient = exc.code in {500, 502, 503, 504}
                compatibility_candidate = (
                    token_field == "max_completion_tokens" and exc.code in {400, 422, 500}
                )
                if transient and attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(f"{request_label}: server error {exc.code}; retrying {attempt+1}/{max_attempts} in {wait:.1f}s...")
                    time.sleep(wait)
                    continue
                if compatibility_candidate:
                    print(f"{request_label}: gateway may not support `max_completion_tokens`; retrying with `max_tokens`.")
                    break
                raise last_error from exc
            except urllib.error.URLError as exc:
                last_error = RuntimeError(f"{request_label} connection error.\nEndpoint: {url!r}\nError: {exc}")
                if attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(f"{request_label}: connection issue; retrying {attempt+1}/{max_attempts} in {wait:.1f}s...")
                    time.sleep(wait)
                    continue
                raise last_error from exc
    raise last_error or RuntimeError(f"{request_label} request failed for an unknown reason.")


def _extract_message_content(data: dict) -> str:
    try:
        content = data["choices"][0]["message"]["content"]
    except (KeyError, IndexError, TypeError) as exc:
        raise ValueError(
            "Unexpected Azure response structure:\n"
            + json.dumps(data, indent=2, ensure_ascii=False)[:3000]
        ) from exc
    if not content:
        raise ValueError(
            "Azure returned an empty message content. Response:\n"
            + json.dumps(data, indent=2, ensure_ascii=False)[:3000]
        )
    return content


def call_writer_llm(system_prompt: str, user_prompt: str) -> str:
    data = _azure_chat_completion(
        url=AZURE_OPENAI_WRITER_URL,
        api_key=AZURE_OPENAI_WRITER_API_KEY,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=2400,
        use_max_completion_tokens=True,
        temperature=None,
        request_label="GPT-5.1 writer",
    )
    return _extract_message_content(data)


def _extract_json_object(text: str) -> str:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*```$", "", text)
    first = text.find("{")
    last = text.rfind("}")
    if first >= 0 and last > first:
        return text[first:last + 1]
    return text


def call_judge_llm_json(system_prompt: str, user_prompt: str) -> dict:
    data = _azure_chat_completion(
        url=AZURE_OPENAI_JUDGE_URL,
        api_key=AZURE_OPENAI_JUDGE_API_KEY,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=2800,
        use_max_completion_tokens=True,
        temperature=None,
        json_mode=True,
        request_label="GPT-5.2 judge",
    )
    content = _extract_message_content(data)
    candidate = _extract_json_object(content)
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        print("GPT-5.2 judge returned incomplete/invalid JSON. Attempting JSON repair...")
        repair_system = (
            "You repair malformed or truncated JSON. "
            "Return one complete valid JSON object only. "
            "Preserve original scores, checklist values, issues and fixes. "
            "Keep strings concise. Do not add markdown fences or commentary."
        )
        repair_user = (
            "Repair the following malformed or truncated judge output into one complete valid JSON object.\n"
            "Requirements:\n"
            "- Keep the same top-level fields when present.\n"
            "- Finish incomplete strings and arrays conservatively.\n"
            "- Limit each issue/fix string to at most 35 words.\n"
            "- Limit arrays to the 6 most important items.\n"
            "- Return JSON only.\n\n"
            f"MALFORMED OUTPUT:\n{content}"
        )
        repaired_data = _azure_chat_completion(
            url=AZURE_OPENAI_JUDGE_URL,
            api_key=AZURE_OPENAI_JUDGE_API_KEY,
            messages=[
                {"role": "system", "content": repair_system},
                {"role": "user", "content": repair_user},
            ],
            max_output_tokens=2400,
            use_max_completion_tokens=True,
            temperature=None,
            json_mode=True,
            request_label="GPT-5.2 judge JSON repair",
        )
        repaired_content = _extract_message_content(repaired_data)
        repaired_candidate = _extract_json_object(repaired_content)
        try:
            return json.loads(repaired_candidate)
        except json.JSONDecodeError as exc:
            raise ValueError(
                "GPT-5.2 judge failed to return valid JSON even after repair.\n"
                f"Original: {content[:4000]}\nRepair: {repaired_content[:4000]}"
            ) from exc


def call_reviser_llm(system_prompt: str, user_prompt: str) -> str:
    data = _azure_chat_completion(
        url=AZURE_OPENAI_REVISER_URL,
        api_key=AZURE_OPENAI_REVISER_API_KEY,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=2400,
        use_max_completion_tokens=False,
        temperature=0.1,
        request_label="GPT-4.1 reviser",
    )
    return _extract_message_content(data)


print("Role-specific LLM helper functions ready")


# ============================================================================
# NOTEBOOK CODE CELL 4
# ============================================================================
# ── LOAD SECTION-SPECIFIC PAYLOADS ───────────────────────────

def find_payload_file(filename: str):
    search_dirs = [
        Path.cwd(),
        Path.cwd() / "Data",
        Path.cwd() / "payloads",
        Path.cwd().parent / "payloads",
        Path.cwd().parent / "Data",
        Path("/mnt/data"),
    ]
    for base in search_dirs:
        candidate = base / filename
        if candidate.exists():
            return candidate
    return None


def load_json_file(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


GOVERNANCE_PAYLOAD_PATH = (
    find_payload_file("payload_BANK01_governance.json")
    or find_payload_file("payload_BANK01.json")
)
STRATEGY_PAYLOAD_PATH = (
    find_payload_file("payload_BANK01_strategy.json")
    or find_payload_file("payload_BANK01.json")
)

if GOVERNANCE_PAYLOAD_PATH is None:
    raise FileNotFoundError("Could not find payload_BANK01_governance.json or payload_BANK01.json.")
if STRATEGY_PAYLOAD_PATH is None:
    raise FileNotFoundError("Could not find payload_BANK01_strategy.json or payload_BANK01.json.")

governance_payload = load_json_file(GOVERNANCE_PAYLOAD_PATH)
strategy_payload   = load_json_file(STRATEGY_PAYLOAD_PATH)

payload    = governance_payload
PAYLOAD_PATH = GOVERNANCE_PAYLOAD_PATH
bank_name  = governance_payload["bank"]["bank_name"]

print(f"Loaded Governance payload for: {bank_name}")
print(f"Governance top-level keys: {list(governance_payload.keys())}")
print(f"Strategy top-level keys:   {list(strategy_payload.keys())}")


# ============================================================================
# NOTEBOOK CODE CELL 5
# ============================================================================
# ── GOVERNANCE EVIDENCE EXTRACTOR ────────────────────────────
#
# FIX: The governance payload ALWAYS contains climate_risk_register for this bank.
# The extractor now correctly reflects this, enabling the management responsibility
# subsection to describe the full risk-register process when evidence is present.
# The code remains data-driven: it checks at runtime whether the register is present.

def _is_present(value) -> bool:
    return value is not None and str(value).strip().lower() not in {"", "nan", "none", "null"}


def _safe_int(value, default=0):
    try:
        return int(value)
    except Exception:
        return default


def _normalise_text(value) -> str:
    return re.sub(r"\s+", " ", str(value or "")).strip()


def extract_management_process_evidence(payload: dict, year: int = 2024) -> dict:
    """
    Summarise Management Responsibility evidence.

    Preferred source: climate_risk_register when present for the reporting year.
    Fallback source:  governance table fields.
    """
    risks = [
        r for r in payload.get("climate_risk_register", [])
        if isinstance(r, dict) and _safe_int(r.get("reporting_year")) == year
    ]

    gov_records = payload.get("governance", [])
    gov_2024 = next(
        (r for r in gov_records if isinstance(r, dict) and _safe_int(r.get("reporting_year")) == year),
        {},
    )

    governance_controls_available = any(
        _is_present(gov_2024.get(field))
        for field in [
            "management_committee_name",
            "climate_risk_reporting_to_board",
            "erm_integration_flag",
            "major_transactions_climate_check",
        ]
    )

    if not risks:
        return {
            "risk_register_available": False,
            "governance_controls_available": governance_controls_available,
            "reporting_year": year,
            "management_committee_name": gov_2024.get("management_committee_name"),
            "climate_risk_reporting_to_board": gov_2024.get("climate_risk_reporting_to_board"),
            "erm_integration_flag": gov_2024.get("erm_integration_flag"),
            "major_transactions_climate_check": gov_2024.get("major_transactions_climate_check"),
            "formal_escalation_thresholds_available": False,
            "message": (
                "The Governance payload does not contain climate_risk_register records. "
                "Management Responsibility can be described only using governance-level evidence."
            ),
            "process_flow_instruction": (
                "Do not claim a risk-register workflow, risk counts, risk categories, scenario links, "
                "monitoring frequencies by risk, or mitigation actions unless risk-register evidence is present. "
                "Use the available governance controls only: management committee name, board reporting frequency, "
                "ERM integration flag and major-transaction climate check."
            ),
        }

    frequencies = sorted({
        str(r.get("monitoring_frequency"))
        for r in risks
        if _is_present(r.get("monitoring_frequency"))
    })
    risk_categories = sorted({
        str(r.get("risk_category"))
        for r in risks
        if _is_present(r.get("risk_category"))
    })
    risk_ratings = sorted({
        str(r.get("risk_rating"))
        for r in risks
        if _is_present(r.get("risk_rating"))
    })
    scenario_links = sorted({
        str(r.get("scenario_analysis_link"))
        for r in risks
        if _is_present(r.get("scenario_analysis_link"))
    })
    mitigation_actions = sorted({
        str(r.get("mitigation_actions"))
        for r in risks
        if _is_present(r.get("mitigation_actions"))
    })

    integrated_count = sum(1 for r in risks if r.get("erm_integrated_flag") is True)
    changed_count = sum(1 for r in risks if r.get("changed_since_prior_period") is True)

    rating_priority = {"critical": 4, "high": 3, "medium": 2, "low": 1}
    sorted_risks = sorted(
        risks,
        key=lambda r: (
            rating_priority.get(str(r.get("risk_rating", "")).lower(), 0),
            float(r.get("financial_impact_meur") or 0),
        ),
        reverse=True,
    )

    material_risk_examples = []
    for r in sorted_risks[:5]:
        material_risk_examples.append({
            "risk_id": r.get("risk_id"),
            "risk_name": r.get("risk_name"),
            "risk_category": r.get("risk_category"),
            "risk_rating": r.get("risk_rating"),
            "time_horizon": r.get("time_horizon"),
            "monitoring_frequency": r.get("monitoring_frequency"),
            "erm_integrated_flag": r.get("erm_integrated_flag"),
            "scenario_analysis_link": r.get("scenario_analysis_link"),
            "mitigation_actions": r.get("mitigation_actions"),
        })

    return {
        "risk_register_available": True,
        "governance_controls_available": governance_controls_available,
        "reporting_year": year,
        "management_committee_name": gov_2024.get("management_committee_name"),
        "climate_risk_reporting_to_board": gov_2024.get("climate_risk_reporting_to_board"),
        "erm_integration_flag": gov_2024.get("erm_integration_flag"),
        "major_transactions_climate_check": gov_2024.get("major_transactions_climate_check"),
        "risk_count": len(risks),
        "erm_integrated_count": integrated_count,
        "changed_since_prior_period_count": changed_count,
        "monitoring_frequencies": frequencies,
        "risk_categories": risk_categories,
        "risk_ratings": risk_ratings,
        "scenario_analysis_links": scenario_links,
        "mitigation_actions": mitigation_actions[:8],
        "material_risk_examples": material_risk_examples,
        "formal_escalation_thresholds_available": False,
        "process_flow_instruction": (
            "Write management responsibility as a process flow: risk identification/register, "
            "classification by category/time horizon/rating, monitoring frequency, scenario links, "
            "mitigation actions and ERM integration. "
            "Do not invent formal escalation thresholds — these are not evidenced."
        ),
    }


def extract_governance_evidence(payload: dict) -> dict:
    gov_records   = payload.get("governance", [])
    board_minutes = payload.get("board_minutes", [])
    bank          = payload.get("bank", {})
    reporting_kpis = payload.get("reporting_kpis", {})
    metadata      = payload.get("metadata", {})

    reporting_year = int(metadata.get("reporting_year", 2024))

    gov_by_year = {
        str(r["reporting_year"]): r
        for r in gov_records
        if isinstance(r, dict) and "reporting_year" in r
    }

    gov_trend = []
    for year in ["2022", "2023", "2024"]:
        if year in gov_by_year:
            g = gov_by_year[year]
            gov_trend.append({
                "year": int(year),
                "esg_committee_meetings": g.get("esg_committee_meetings_per_year"),
                "board_climate_expertise_pct": g.get("board_climate_expertise_pct"),
                "ceo_esg_compensation_pct": g.get("ceo_esg_compensation_pct"),
                "all_exec_climate_remuneration_pct": g.get("all_exec_climate_remuneration_pct"),
                "climate_on_board_agenda_pct": g.get("climate_on_board_agenda_pct"),
                "management_committee_name": g.get("management_committee_name"),
                "board_full_meeting_frequency": g.get("board_full_meeting_frequency"),
            })

    PRIORITY_TOPICS = [
        "scenario", "transition", "net_zero", "target", "carbon_credit",
        "remuneration", "tcfd", "green_finance", "physical_risk", "risk", "esg"
    ]

    def decision_score(m: dict) -> int:
        topics   = str(m.get("climate_topics_discussed", "")).lower()
        decision = str(m.get("decision_summary", "")).lower()
        ifrs     = str(m.get("ifrs_s2_para_evidence", "")).lower()
        score    = sum(1 for t in PRIORITY_TOPICS if t in topics or t in decision)
        if "6(a)(v)" in ifrs:
            score += 2
        if str(m.get("committee_type", "")).lower() == "full_board":
            score += 1
        return score

    minutes_year = [
        m for m in board_minutes
        if isinstance(m, dict)
        and _safe_int(m.get("reporting_year")) == reporting_year
        and m.get("decision_made_flag") is True
        and _is_present(m.get("decision_summary"))
        and _is_present(m.get("meeting_id"))
    ]

    # ── FIX: committee-aware deduplication ───────────────────────────────────
    # The original code collapsed records by decision_summary text alone,
    # which merges Full Board and ESG Committee approvals of identically-worded
    # decisions (e.g. "Approved carbon credit procurement budget" appeared 4 times
    # across both committee types). The winning record's topics_discussed then
    # misrepresented the committee context the writer used for the oversight narrative.
    #
    # New logic: deduplicate within each committee_type separately, then merge the
    # best record per (decision_text, committee_type) pair. This preserves the
    # distinct governance layer (Full Board vs ESG Committee) for each decision
    # while still removing true same-committee duplicates.
    # ─────────────────────────────────────────────────────────────────────────
    best_by_decision_committee = {}
    for m in minutes_year:
        decision_text  = re.sub(r"\s+", " ", str(m.get("decision_summary", "")).lower().strip())
        committee_type = str(m.get("committee_type", "")).lower().strip()
        key = (decision_text, committee_type)
        current = best_by_decision_committee.get(key)
        if current is None or decision_score(m) > decision_score(current):
            best_by_decision_committee[key] = m

    selected_decisions = []
    for m in sorted(best_by_decision_committee.values(), key=decision_score, reverse=True)[:6]:
        selected_decisions.append({
            "meeting_id": m.get("meeting_id"),
            "date": m.get("meeting_date"),
            "committee": m.get("committee_name"),
            "committee_type": m.get("committee_type"),
            "topics_discussed": m.get("climate_topics_discussed"),
            "decision": m.get("decision_summary"),
            "ifrs_evidence_para": m.get("ifrs_s2_para_evidence"),
            "internal_ref": f"[REF:{m.get('meeting_id')}]",
        })

    gov_2024 = gov_by_year.get(str(reporting_year), {})

    governance_instrument_fields = [
        "committee_charter", "committee_terms_of_reference", "board_mandate",
        "esg_committee_mandate", "formal_climate_mandate",
        "governance_policy_reference", "committee_charter_climate_mandate",
    ]
    formal_mandate_available = any(_is_present(gov_2024.get(f)) for f in governance_instrument_fields)

    tradeoff_terms = [
        "tradeoff", "trade-off", "capital allocation", "profitability",
        "implementation cost", "risk appetite", "competing priority", "competing priorities"
    ]
    tradeoff_decisions = [
        m for m in minutes_year
        if any(
            term in str(m.get("decision_summary", "")).lower()
            or term in str(m.get("climate_topics_discussed", "")).lower()
            for term in tradeoff_terms
        )
    ]
    board_tradeoff_evidence_available = len(tradeoff_decisions) > 0

    skills_process_fields = [
        "skills_matrix", "skills_assessment_process", "board_skills_review",
        "skills_adequacy_assessment", "director_training_frequency",
        "training_hours", "skills_gap_analysis",
    ]
    skills_adequacy_process_available = any(_is_present(gov_2024.get(f)) for f in skills_process_fields)

    assurance_scope = str(gov_2024.get("assurance_scope", ""))
    financed_emissions_2024 = reporting_kpis.get("financed_emissions_2024_tco2e")
    financed_in_scope = "financed" in assurance_scope.lower() or "scope 3" in assurance_scope.lower()

    assurance_scope_limitation = {
        "assurance_scope":    assurance_scope,
        "external_assurance": gov_2024.get("external_assurance"),
        "provider":           gov_2024.get("assurance_provider"),
        "standard":           gov_2024.get("assurance_standard"),
        "financed_emissions_2024_tco2e": financed_emissions_2024,
        "financed_emissions_in_scope":   financed_in_scope,
        "instruction": (
            "State that assurance covers only the stated scope. If the stated scope is Scope 1 and 2 emissions, "
            "do not imply financed emissions or other Scope 3 categories are assured. For a bank, explicitly clarify "
            "that financed emissions are outside the stated assurance scope based on available evidence."
        ),
    }

    management_evidence = extract_management_process_evidence(payload, year=reporting_year)

    # ── Distinct committee-type summary for the writer ───────────────────────
    # Gives the writer clear facts about which bodies made which types of decisions
    # so it can accurately describe committee roles without conflating topics.
    committee_decision_summary = {}
    for m in selected_decisions:
        ctype = m.get("committee_type", "unknown")
        if ctype not in committee_decision_summary:
            committee_decision_summary[ctype] = {
                "committee_name": m.get("committee"),
                "decisions": [],
                "topics": set(),
            }
        committee_decision_summary[ctype]["decisions"].append(m.get("decision"))
        for t in str(m.get("topics_discussed") or "").split("|"):
            if t.strip():
                committee_decision_summary[ctype]["topics"].add(t.strip())
    # Convert sets to sorted lists for JSON serialisation
    for v in committee_decision_summary.values():
        v["topics"] = sorted(v["topics"])

    return {
        "bank": {
            "name":              bank.get("bank_name"),
            "bank_id":           bank.get("bank_id"),
            "country":           bank.get("country"),
            "total_assets_meur": bank.get("total_assets_meur"),
            "regulatory_regime": bank.get("regulatory_regime"),
        },
        "reporting_year":   reporting_year,
        "comparative_years": metadata.get("comparative_years", [2022, 2023]),
        "payload_profile": {
            "source_payload": "governance",
            "top_level_keys": list(payload.keys()),
            "climate_risk_register_in_payload": "climate_risk_register" in payload,
            "board_minutes_count":    len(board_minutes),
            "governance_record_count": len(gov_records),
        },
        "governance_2024": {
            "board_size":                    gov_2024.get("board_size"),
            "independent_directors_pct":     gov_2024.get("independent_directors_pct"),
            "esg_committee_exists":          gov_2024.get("esg_committee_exists"),
            "esg_committee_meetings_per_year": gov_2024.get("esg_committee_meetings_per_year"),
            "board_climate_expertise_pct":   gov_2024.get("board_climate_expertise_pct"),
            "ceo_compensation_esg_linked":   gov_2024.get("ceo_compensation_esg_linked"),
            "ceo_esg_compensation_pct":      gov_2024.get("ceo_esg_compensation_pct"),
            "all_exec_climate_remuneration_pct": gov_2024.get("all_exec_climate_remuneration_pct"),
            "climate_risk_reporting_to_board": gov_2024.get("climate_risk_reporting_to_board"),
            "climate_on_board_agenda_pct":   gov_2024.get("climate_on_board_agenda_pct"),
            "board_full_meeting_frequency":  gov_2024.get("board_full_meeting_frequency"),
            "management_committee_name":     gov_2024.get("management_committee_name"),
            "erm_integration_flag":          gov_2024.get("erm_integration_flag"),
            "skills_development_programme":  gov_2024.get("skills_development_programme"),
            "major_transactions_climate_check": gov_2024.get("major_transactions_climate_check"),
            "external_assurance":  gov_2024.get("external_assurance"),
            "assurance_provider":  gov_2024.get("assurance_provider"),
            "assurance_scope":     gov_2024.get("assurance_scope"),
            "assurance_standard":  gov_2024.get("assurance_standard"),
            "tcfd_aligned":        gov_2024.get("tcfd_aligned"),
            "ifrs_s2_aligned":     gov_2024.get("ifrs_s2_aligned"),
        },
        "governance_trend": gov_trend,
        "management_process_evidence": management_evidence,
        "board_decisions_2024": selected_decisions,
        # ── NEW: committee-type decision summary for writer accuracy ──────────
        "committee_decision_summary": committee_decision_summary,
        "assurance_context": {
            "financed_emissions_2024_tco2e": financed_emissions_2024,
            "assurance_scope":              assurance_scope,
            "financed_emissions_in_scope":   financed_in_scope,
        },
        "strict_governance_evidence": {
            "formal_governance_mandate_available": formal_mandate_available,
            "formal_governance_mandate_instruction": (
                "Do not claim the ESG & Sustainability Committee has a formal climate mandate unless charter "
                "or terms-of-reference evidence is provided. If no formal instrument is available, say that "
                "available documentation evidences committee activity and meeting frequency but does not include "
                "the committee charter or terms of reference."
            ),
            "board_tradeoff_evidence_available": board_tradeoff_evidence_available,
            "tradeoff_decisions": tradeoff_decisions[:3],
            "board_tradeoff_instruction": (
                "Discuss board trade-offs only if explicit trade-off evidence exists in the decision record. "
                "If not, state that the board decision evidence identifies climate-related decisions but does "
                "not describe specific trade-offs such as profitability, capital allocation, implementation "
                "cost, risk appetite or competing strategic priorities."
            ),
            "skills_adequacy_process_available": skills_adequacy_process_available,
            "skills_adequacy_instruction": (
                "Use the board climate expertise percentage and skills development programme as outcome/activity evidence. "
                "Do not invent a formal skills adequacy assessment process."
            ),
            "assurance_scope_limitation": assurance_scope_limitation,
        },
        "interpretation_notes": {
            "climate_on_board_agenda_pct": (
                "This figure represents the percentage of board meetings during the year where climate-related topics "
                "appeared on the agenda. It does not mean percentage of agenda time devoted to climate."
            ),
            "management_committee_names": (
                "Committee names are recorded by year only. The evidence does not prove that one committee evolved into, "
                "replaced, or was renamed as another. State the 2024 committee name and, if comparative names are used, "
                "present them neutrally."
            ),
            "major_transactions_climate_check": (
                "A true value indicates evidence of climate checks for major transactions; it does not prove a formal mandatory policy."
            ),
            "board_decision_traceability": (
                "Each selected decision includes a meeting_id for audit traceability. Meeting IDs must not be printed "
                "in the final report."
            ),
            "committee_decision_accuracy": (
                "Use committee_decision_summary to determine which topics were discussed at which body type. "
                "Do NOT describe a topic as an ESG Committee topic if the only evidence record for it belongs "
                "to the full_board committee_type, and vice versa. "
                "Each decision record includes committee_type and topics_discussed — both must be respected."
            ),
            "avoid_duplication": (
                "Do not list the same board decisions twice. Board oversight should summarise decision governance; "
                "the detailed dated list belongs only in the Board and committee decisions subsection."
            ),
        },
    }


evidence = extract_governance_evidence(governance_payload)

print(f"Evidence extracted for: {evidence['bank']['name']}")
print(f"Governance payload risk register present: {evidence['payload_profile']['climate_risk_register_in_payload']}")
print(f"Board decisions selected: {len(evidence['board_decisions_2024'])}")
print(f"Management evidence risk register available: {evidence['management_process_evidence'].get('risk_register_available')}")
print(f"Management risk count 2024: {evidence['management_process_evidence'].get('risk_count')}")
print("Committee decision summary:")
for ctype, summary in evidence["committee_decision_summary"].items():
    print(f"  {ctype}: decisions={summary['decisions']}, topics={summary['topics']}")
print("Selected decisions with internal refs:")
for d in evidence["board_decisions_2024"]:
    print(f"  {d['date']} | {d['committee']} | {d['decision']} | topics: {d['topics_discussed']}")


# ============================================================================
# NOTEBOOK CODE CELL 6
# ============================================================================
# ── GOVERNANCE EVIDENCE AVAILABILITY + SAVING ────────────────

def build_governance_availability_profile(evidence: dict) -> dict:
    gov       = evidence.get("governance_2024", {})
    trend     = evidence.get("governance_trend", [])
    management = evidence.get("management_process_evidence", {})
    strict    = evidence.get("strict_governance_evidence", {})
    decisions = evidence.get("board_decisions_2024", [])

    def present(value) -> bool:
        return value is not None and str(value).strip().lower() not in {"", "none", "null", "nan"}

    trend_metrics = {
        "esg_committee_meetings": [
            row.get("esg_committee_meetings") for row in trend
            if present(row.get("esg_committee_meetings"))
        ],
        "board_climate_expertise_pct": [
            row.get("board_climate_expertise_pct") for row in trend
            if present(row.get("board_climate_expertise_pct"))
        ],
        "ceo_esg_compensation_pct": [
            row.get("ceo_esg_compensation_pct") for row in trend
            if present(row.get("ceo_esg_compensation_pct"))
        ],
        "all_exec_climate_remuneration_pct": [
            row.get("all_exec_climate_remuneration_pct") for row in trend
            if present(row.get("all_exec_climate_remuneration_pct"))
        ],
        "climate_on_board_agenda_pct": [
            row.get("climate_on_board_agenda_pct") for row in trend
            if present(row.get("climate_on_board_agenda_pct"))
        ],
        "board_full_meeting_frequency": [
            row.get("board_full_meeting_frequency") for row in trend
            if present(row.get("board_full_meeting_frequency"))
        ],
    }

    assurance_scope      = str(gov.get("assurance_scope") or "").lower()
    financed_emissions   = evidence.get("assurance_context", {}).get("financed_emissions_2024_tco2e")
    risk_register_available    = bool(management.get("risk_register_available"))
    governance_controls_available = bool(management.get("governance_controls_available"))

    return {
        "board_core_metrics_available": all(
            present(gov.get(field))
            for field in [
                "board_size",
                "independent_directors_pct",
                "esg_committee_meetings_per_year",
                "climate_risk_reporting_to_board",
                "climate_on_board_agenda_pct",
            ]
        ),
        "trend_metrics_available": {
            name: len(values) >= 2
            for name, values in trend_metrics.items()
        },
        "selected_decision_count": len(decisions),
        "board_decisions_available": len(decisions) > 0,
        "formal_governance_mandate_available": bool(strict.get("formal_governance_mandate_available")),
        "board_tradeoff_evidence_available": bool(strict.get("board_tradeoff_evidence_available")),
        "management_process_available": risk_register_available or governance_controls_available,
        "management_risk_register_available": risk_register_available,
        "management_governance_controls_available": governance_controls_available,
        "formal_escalation_thresholds_available": bool(management.get("formal_escalation_thresholds_available", False)),
        "skills_outcome_metrics_available": present(gov.get("board_climate_expertise_pct")),
        "skills_development_programme_available": bool(gov.get("skills_development_programme")),
        "skills_adequacy_process_available": bool(strict.get("skills_adequacy_process_available")),
        "remuneration_evidence_available": (
            present(gov.get("ceo_esg_compensation_pct"))
            and present(gov.get("all_exec_climate_remuneration_pct"))
        ),
        "assurance_evidence_available": all(
            present(gov.get(field))
            for field in ["external_assurance", "assurance_provider", "assurance_scope", "assurance_standard"]
        ),
        "assurance_scope_is_scope1_scope2_only": (
            "scope 1" in assurance_scope and "scope 2" in assurance_scope
        ),
        "financed_emissions_available_for_scope_context": present(financed_emissions),
        "payload_boundary": {
            "governance_payload_has_climate_risk_register": evidence.get("payload_profile", {}).get("climate_risk_register_in_payload"),
            "governance_payload_tables": evidence.get("payload_profile", {}).get("top_level_keys", []),
        },
        "writer_policy": {
            "use_available_evidence": (
                "Use every material governance evidence item that is available and relevant."
            ),
            "handle_unavailable_evidence": (
                "When a material governance requirement is not supported by the available evidence, state the boundary "
                "once in the relevant subsection. Do not invent the missing process or control."
            ),
            "management_process_boundary": (
                "If climate_risk_register IS available, describe the full risk-register process. "
                "If absent, describe only the management committee, board reporting frequency, ERM integration flag "
                "and major-transaction climate check."
            ),
            "wording": (
                "Use 'available evidence', 'available documentation', or 'source data' in final disclosure; "
                "do not use the word 'payload'."
            ),
        },
    }


def add_governance_traceability(evidence: dict, source_payload_path: Path) -> dict:
    evidence = dict(evidence)
    evidence["availability_profile"] = build_governance_availability_profile(evidence)
    source_tables = ["bank", "governance", "board_minutes", "reporting_kpis"]
    if evidence.get("payload_profile", {}).get("climate_risk_register_in_payload"):
        source_tables.append("climate_risk_register")
    evidence["source_traceability"] = {
        "source_payload_path": str(source_payload_path),
        "source_tables": source_tables,
        "selected_board_decision_refs": [
            item.get("internal_ref")
            for item in evidence.get("board_decisions_2024", [])
            if item.get("internal_ref")
        ],
        "management_risk_refs": [
            item.get("risk_id")
            for item in evidence.get("management_process_evidence", {}).get("material_risk_examples", [])
            if item.get("risk_id")
        ],
    }
    return evidence


evidence = add_governance_traceability(evidence, GOVERNANCE_PAYLOAD_PATH)

raw_governance_payload = {
    key: governance_payload.get(key)
    for key in ["metadata", "bank", "governance", "board_minutes", "climate_risk_register", "reporting_kpis"]
    if key in governance_payload
}

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

RAW_GOVERNANCE_PATH     = output_dir / "payload_BANK01_governance_raw.json"
COMPACT_GOVERNANCE_PATH = output_dir / "compact_governance_evidence_BANK01.json"

with open(RAW_GOVERNANCE_PATH, "w", encoding="utf-8") as f:
    json.dump(raw_governance_payload, f, indent=2, ensure_ascii=False)
with open(COMPACT_GOVERNANCE_PATH, "w", encoding="utf-8") as f:
    json.dump(evidence, f, indent=2, ensure_ascii=False)

print("Governance evidence prepared")
print(f"- management_risk_register_available: {evidence['availability_profile']['management_risk_register_available']}")
print(json.dumps(evidence["availability_profile"], indent=2, ensure_ascii=False))


# ============================================================================
# NOTEBOOK CODE CELL 7
# ============================================================================
class GovernanceState(TypedDict):
    bank_name:       str
    evidence:        dict
    draft:           str
    judge_result:    dict
    revision_count:  int
    max_revisions:   int
    status:          Literal["drafting", "judging", "revising", "approved", "failed"]
    final_section:   str
    token_usage:     dict


# ============================================================================
# NOTEBOOK CODE CELL 8
# ============================================================================
# ── GOVERNANCE REQUIREMENTS ─────────────────────────────────

IFRS_GOVERNANCE_REQUIREMENTS = """
STRICT GOVERNANCE DISCLOSURE REQUIREMENTS FOR THIS SECTION:

Final section title and headings must be exactly:
### Governance
#### Board oversight
#### Management responsibility
#### Climate skills and competencies
#### Remuneration and climate incentives
#### Board and committee decisions during 2024
#### External assurance and controls

Data-aware coverage requirements:
1. Use the Governance payload as the source of governance facts.
2. Use only the available Governance evidence: board composition, committee activity, climate agenda
   frequency, board reporting frequency, remuneration metrics, board decisions, assurance scope and
   available management governance controls.
3. If climate_risk_register is PRESENT in the Governance evidence, Management responsibility MUST
   describe the full risk-register process: risk identification, classification by category/time
   horizon/rating, monitoring frequency, scenario links, mitigation actions and ERM integration.
4. If climate_risk_register is ABSENT, Management responsibility must be limited to:
   - management committee name;
   - climate risk reporting frequency to the board;
   - ERM integration flag;
   - major-transaction climate check flag.
5. Do not invent formal committee charters, terms of reference, formal governance mandates,
   formal escalation thresholds, board trade-off analysis or formal skills adequacy assessment.
6. Use year-on-year trends only for metrics present across comparative years.
7. Use board decisions only from the 2024 selected decisions. Put the detailed dated list ONLY in
   the dedicated decisions subsection. Do NOT repeat those decisions in Board oversight.
8. For each decision, use its own committee_type and topics_discussed fields.
   Do NOT describe a topic as an ESG Committee topic if the record belongs to full_board type,
   and vice versa. Use the committee_decision_summary to verify which topics belong to which body.
9. Describe assurance only for the stated scope. If assurance scope is Scope 1 and 2 emissions,
   state that financed emissions / Scope 3 are outside the stated assurance scope.
10. Do not include visible IFRS paragraph references, meeting IDs, internal refs or bracketed IFRS
    tags in the final output.
11. Use "available evidence", "available documentation" or "source data" in limitation wording.
    Do not use the technical word "payload" in the final report.
""".strip()

print("Governance requirements ready")


# ============================================================================
# NOTEBOOK CODE CELL 9
# ============================================================================
# ── GOVERNANCE WRITER PROMPT ─────────────────────────────────

WRITER_SYSTEM = """
You are a senior sustainability reporting specialist writing the Governance section of an
IFRS S1/S2-aligned climate disclosure report for a commercial bank.

Write in a formal, third-person, publication-ready style.

Evidence rules:
- Use only the compact Governance evidence supplied by the user prompt.
- Do not invent missing governance policies, committee charters, trade-offs, escalation thresholds,
  or skills assessment processes.
- When evidence is missing, state the limitation once in report-style language using "available
  evidence", "available documentation" or "source data".
- Do not use the word "payload" in the final section.
- Do not include IFRS paragraph references, meeting IDs, internal references or bracketed evidence tags.
- Do not use markdown tables.

Output rules:
- Return only the complete Governance section.
- Keep exactly the six required subsections.
""".strip()


def build_writer_prompt(evidence: dict, judge_feedback: str = None) -> str:
    profile    = evidence.get("availability_profile", {})
    management = evidence.get("management_process_evidence", {})
    strict     = evidence.get("strict_governance_evidence", {})
    committee_summary = evidence.get("committee_decision_summary", {})

    instructions = []

    instructions.append(
        "- Use board size, independent-director percentage, full-board meeting count, "
        "climate agenda percentage, committee meeting count and board reporting frequency."
    )

    if profile.get("formal_governance_mandate_available"):
        instructions.append("- Describe the formal governance mandate using the supplied direct evidence.")
    else:
        instructions.append(
            "- Committee activity is evidenced, but no committee charter/terms of reference/formal mandate "
            "is available. State this boundary once in Board oversight."
        )

    if profile.get("board_tradeoff_evidence_available"):
        instructions.append("- Describe only the documented board trade-offs included in evidence.")
    else:
        instructions.append(
            "- Board decisions are evidenced, but specific trade-offs are not documented. "
            "State this boundary once without inventing trade-offs."
        )

    # ── FIX: correct management instruction based on actual risk register presence ──
    if profile.get("management_risk_register_available"):
        instructions.append(
            "- Management responsibility MUST describe the full risk-register process using the supplied "
            "material_risk_examples: risk identification, classification by category/time horizon/rating, "
            "monitoring frequency, scenario links, mitigation actions and ERM integration. "
            "The Climate Risk Management Committee oversees this process and reports semi-annually to the Board."
        )
    elif profile.get("management_governance_controls_available"):
        instructions.append(
            "- Management responsibility must NOT describe a risk-register workflow. Use only the Climate Risk "
            "Management Committee, semi-annual board reporting, ERM integration flag and major-transaction "
            "climate check."
        )
    else:
        instructions.append(
            "- Management process evidence is not available. State the boundary without inventing a process."
        )

    if not profile.get("formal_escalation_thresholds_available"):
        instructions.append(
            "- Formal escalation thresholds are not evidenced. "
            "Do not imply a formal trigger-based escalation design. "
            "You may describe the reporting channel (committee → Board semi-annual) without implying thresholds."
        )

    if profile.get("skills_adequacy_process_available"):
        instructions.append("- Describe the formal board skills adequacy assessment process from evidence.")
    else:
        instructions.append(
            "- Use board climate expertise percentage and skills development programme only. "
            "State that no formal board skills adequacy assessment process is documented."
        )

    if profile.get("remuneration_evidence_available"):
        instructions.append(
            "- Use both CEO ESG-linked remuneration percentage and all-executive climate-linked remuneration percentage."
        )

    if profile.get("assurance_evidence_available"):
        instructions.append(
            "- Describe assurance using exact provider, level, standard and scope."
        )
        if profile.get("assurance_scope_is_scope1_scope2_only"):
            instructions.append(
                "- Clarify that assurance covers Scope 1 and Scope 2 emissions only, and financed emissions / "
                "Scope 3 are outside the stated assurance scope."
            )

    # ── FIX: inject committee decision summary to prevent cross-body topic confusion ──
    committee_accuracy_note = (
        "\nCOMMITTEE DECISION ACCURACY:\n"
        "The following is a pre-computed summary of which decisions and topics belong to each committee body. "
        "Use ONLY this mapping when describing committee roles in Board oversight. "
        "Do not attribute a topic to a committee type if it does not appear under that type below.\n"
        f"{json.dumps(committee_summary, indent=2, ensure_ascii=False)}"
    )

    feedback_block = ""
    if judge_feedback:
        feedback_block = (
            "JUDGE FEEDBACK TO ADDRESS:\n"
            f"{judge_feedback}\n\n"
            "REVISION RULES:\n"
            "- Fix the judge's valid issues using available evidence.\n"
            "- Preserve correct evidence-boundary statements.\n"
            "- Never invent unavailable information."
        )

    return f"""
{IFRS_GOVERNANCE_REQUIREMENTS}

BANK:
{evidence['bank']['name']} ({evidence['bank']['country']})
REPORTING YEAR: {evidence['reporting_year']}
COMPARATIVE YEARS: {evidence['comparative_years']}

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(profile, indent=2, ensure_ascii=False)}

DATA-AWARE WRITING INSTRUCTIONS:
{chr(10).join(instructions)}

{committee_accuracy_note}

COMPACT GOVERNANCE EVIDENCE — USE ONLY THIS DATA:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

IMPORTANT DATA BOUNDARIES:
- Governance payload tables available: {evidence.get('payload_profile', {}).get('top_level_keys')}
- Climate risk register present in Governance payload: {evidence.get('payload_profile', {}).get('climate_risk_register_in_payload')}
- Management process instruction: {management.get('process_flow_instruction')}
- Formal mandate instruction: {strict.get('formal_governance_mandate_instruction')}
- Board trade-off instruction: {strict.get('board_tradeoff_instruction')}
- Skills instruction: {strict.get('skills_adequacy_instruction')}
- Assurance instruction: {strict.get('assurance_scope_limitation', {}).get('instruction')}

GENERAL WRITING RULES:
- Do not duplicate the detailed board-decision list in Board oversight.
- Use dates and decision descriptions only in the dedicated decisions subsection.
- Interpret climate_on_board_agenda_pct as the percentage of board meetings where climate appeared on the agenda.
- A true major_transactions_climate_check flag indicates evidence of climate checks; it does not prove a formal mandatory policy.
- Do not infer that committee names from 2022, 2023 and 2024 represent the same renamed committee.
- Do not imply formal escalation thresholds or trigger-based escalation mechanics.
- TCFD-aligned and IFRS S2-aligned flags indicate alignment status; governance oversight does not itself 'support' alignment — reframe as: the bank's source data flags the disclosures as TCFD-aligned and IFRS S2-aligned.

{feedback_block}

Write the complete Governance section now.
Return only the final Governance section.
""".strip()


# ============================================================================
# NOTEBOOK CODE CELL 10
# ============================================================================
# ── GOVERNANCE JUDGE PROMPT ─────────────────────────────────

JUDGE_SYSTEM = """
You are a strict sustainability-reporting judge for an IFRS S1/S2-aligned bank climate disclosure.

You evaluate whether the Governance section is:
- supported by the compact evidence;
- aligned with the Governance disclosure checklist;
- transparent about missing evidence;
- free from hallucinated policies, processes, trade-offs, assurance coverage and unsupported governance claims.

Return valid JSON only.
""".strip()


def build_judge_prompt(draft: str, evidence: dict, deterministic_checks: dict = None) -> str:
    profile = evidence.get("availability_profile", {})
    deterministic_checks = deterministic_checks or {}

    return f"""
Evaluate the Governance draft against the compact evidence, availability profile and deterministic pre-checks.

DRAFT:
{draft}

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(profile, indent=2, ensure_ascii=False)}

DETERMINISTIC PRE-CHECKS:
{json.dumps(deterministic_checks, indent=2, ensure_ascii=False)}

COMPACT GOVERNANCE EVIDENCE:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

EVALUATION PRINCIPLES:
1. Penalise any claim that contradicts evidence or invents unavailable governance information.
2. Penalise omission when material evidence is available but not used.
3. Do not demand unavailable evidence. Correctly disclosed evidence boundaries are acceptable but lower completeness.
4. Distinguish committee activity evidence from formal charter/mandate evidence.
5. If climate_risk_register IS present in the Governance evidence, the draft MUST describe the risk-register process.
   Failure to use the available risk register is an evidence omission.
6. If climate_risk_register IS present, the draft may use risk counts, risk categories, scenario links,
   monitoring frequencies and mitigation actions — these are all evidenced.
7. If formal mandate, board trade-offs, formal escalation thresholds or skills adequacy assessment are unavailable,
   the draft must not invent them.
8. For each decision in board_decisions_2024, verify the draft uses the correct committee_type and topics_discussed.
   A topic is correctly attributed only if it appears under that committee_type in committee_decision_summary.
9. Verify exact figures: 10 board members, 68.5% independence, 72.6% climate agenda frequency, 7 ESG committee
   meetings, 37.5% climate expertise, 9.3% CEO ESG remuneration and 15.1% all-executive climate remuneration.
10. Verify that assurance is limited to Scope 1 and Scope 2 emissions and does not imply financed emissions assurance.
11. Verify no visible IFRS paragraph references, meeting IDs or internal refs appear in the final section.
12. Verify decisions are not duplicated between Board oversight and the dedicated decisions subsection.
13. Alignment flag wording: TCFD-aligned and IFRS S2-aligned are source data flags.
    The draft must not claim that governance oversight "supports" alignment — it should say the source data flags it.

SCORING:
- 9–10: Directly evidenced, materially complete, no unsupported claims, no material evidence boundary.
- 8: Strong, with only one material boundary correctly disclosed.
- 7: Usable, with multiple correctly disclosed boundaries.
- 6: Revision required because available evidence is omitted, or unsupported wording remains.
- 5 or below: Major evidence failure, hallucination, contradiction or missing core subsection.

Return valid JSON only. Keep arrays concise:
{{
  "overall_score": <integer 1-10>,
  "evidence_support_score": <integer 1-10>,
  "ifrs_alignment_score": <integer 1-10>,
  "specificity_score": <integer 1-10>,
  "hallucination_risk": "<low|medium|high>",
  "approval_status": "<approved|approved_with_limitations|revision_required|rejected>",
  "approved": <true if approved or approved_with_limitations, otherwise false>,
  "checklist": {{
    "required_structure_present": <true/false>,
    "board_metrics_used_correctly": <true/false>,
    "trend_metrics_used_correctly": <true/false>,
    "board_decisions_used_correctly": <true/false>,
    "committee_topic_attribution_correct": <true/false>,
    "formal_mandate_handled_according_to_availability": <true/false>,
    "board_tradeoffs_handled_according_to_availability": <true/false>,
    "management_evidence_handled_according_to_availability": <true/false>,
    "risk_register_used_when_present": <true/false>,
    "escalation_handled_according_to_availability": <true/false>,
    "skills_handled_according_to_availability": <true/false>,
    "remuneration_evidence_used_correctly": <true/false>,
    "assurance_scope_used_correctly": <true/false>,
    "no_unsupported_committee_evolution": <true/false>,
    "no_duplicate_decisions": <true/false>,
    "no_visible_ifrs_refs_or_internal_refs": <true/false>,
    "no_unsupported_strong_claims": <true/false>,
    "alignment_flag_wording_correct": <true/false>
  }},
  "available_evidence_omitted": [<specific available evidence omitted>],
  "unsupported_claims": [<specific unsupported claims>],
  "correctly_disclosed_evidence_boundaries": [<accurate boundary statements>],
  "main_issues": [<specific issues>],
  "required_fixes": [<actionable evidence-aware fixes>]
}}
""".strip()


# ============================================================================
# NOTEBOOK CODE CELL 12
# ============================================================================
# ── GOVERNANCE LANGGRAPH NODES ───────────────────────────────

def _contains_any(text: str, phrases: list) -> bool:
    text_l = text.lower()
    return any(p.lower() in text_l for p in phrases)


def run_governance_deterministic_checks(draft: str, evidence: dict) -> dict:
    text    = draft or ""
    text_l  = text.lower()
    profile = evidence.get("availability_profile", {})
    gov     = evidence.get("governance_2024", {})

    required_headings = [
        "#### Board oversight",
        "#### Management responsibility",
        "#### Climate skills and competencies",
        "#### Remuneration and climate incentives",
        "#### Board and committee decisions during 2024",
        "#### External assurance and controls",
    ]
    missing_headings = [h for h in required_headings if h not in text]

    warnings = []
    failures = []

    if missing_headings:
        failures.append({"check": "missing_required_headings", "details": missing_headings})

    if re.search(r"IFRS\s*S?[12]?\s*§|§\s*\d|\[IFRS", text):
        failures.append({"check": "visible_ifrs_references", "details": "Visible IFRS paragraph references/tags found."})

    if "[REF:" in text or re.search(r"MTG-BANK\d+", text):
        failures.append({"check": "internal_refs_visible", "details": "Internal meeting references should not appear in final text."})

    if "72.6" not in text:
        warnings.append({"check": "climate_agenda_pct_not_explicit", "details": "72.6% climate agenda frequency may be missing."})

    if "9.3" not in text:
        warnings.append({"check": "ceo_remuneration_pct_not_explicit", "details": "9.3% CEO ESG-linked remuneration may be missing."})

    if "15.1" not in text:
        warnings.append({"check": "executive_remuneration_pct_not_explicit", "details": "15.1% all-executive climate remuneration may be missing."})

    # ── FIX: management risk register check ─────────────────────────────────
    # The governance payload DOES contain climate_risk_register (8 records for 2024).
    # If the draft fails to describe the risk-register process, that is an omission, not an
    # acceptable boundary statement.
    if profile.get("management_risk_register_available"):
        risk_register_terms = ["risk register", "risk categories", "monitoring frequenc", "mitigation action", "scenario link", "erm integrat"]
        if not any(term in text_l for term in risk_register_terms):
            failures.append({
                "check": "risk_register_not_used_when_present",
                "details": (
                    "The governance payload contains climate_risk_register records for 2024. "
                    "Management responsibility must describe the risk-register process. "
                    "This is an evidence omission, not a valid boundary statement."
                ),
            })
    # ─────────────────────────────────────────────────────────────────────────

    if not profile.get("formal_governance_mandate_available"):
        if _contains_any(text, ["formal mandate", "committee charter", "terms of reference"]) and \
           not _contains_any(text, ["does not include", "not documented", "not available", "available documentation does not"]):
            failures.append({"check": "formal_mandate_overclaimed", "details": "Formal mandate/charter language used without a limitation."})

    if not profile.get("board_tradeoff_evidence_available"):
        if _contains_any(text, ["trade-off", "tradeoff", "capital allocation", "profitability", "risk appetite"]) and \
           not _contains_any(text, ["not describe", "not documented", "not available", "does not provide"]):
            failures.append({"check": "board_tradeoffs_overclaimed", "details": "Trade-off language used without direct evidence or limitation."})

    if not profile.get("formal_escalation_thresholds_available"):
        if _contains_any(text, ["escalation threshold", "formal escalation", "escalated when", "escalation route",
                                  "trigger", "triggers escalat", "escalation mechanic"]) and \
           not _contains_any(text, ["not documented", "not available", "does not specify", "not evidenced", "does not evidence"]):
            failures.append({"check": "escalation_threshold_overclaimed", "details": "Formal escalation language appears without evidence."})

    if not profile.get("skills_adequacy_process_available"):
        if _contains_any(text, ["skills adequacy assessment", "skills matrix", "skills gap analysis", "formal skills assessment"]) and \
           not _contains_any(text, ["not documented", "not available", "does not describe"]):
            failures.append({"check": "skills_process_overclaimed", "details": "Formal skills adequacy process appears without evidence."})

    if profile.get("assurance_scope_is_scope1_scope2_only"):
        if "assurance" in text_l and "financed emissions" not in text_l:
            warnings.append({"check": "financed_emissions_scope_boundary_missing", "details": "Assurance section may not clearly state financed emissions are outside assurance scope."})
        if _contains_any(text, ["financed emissions are assured", "scope 3 emissions are assured", "assurance over financed emissions"]):
            failures.append({"check": "assurance_scope_overclaimed", "details": "Draft implies financed emissions / Scope 3 are assured."})

    # ── FIX: alignment flag wording check ───────────────────────────────────
    if _contains_any(text, ["governance oversight supports", "governance supports the bank's tcfd",
                              "oversight supports the bank's ifrs", "supports alignment"]):
        failures.append({
            "check": "alignment_flag_wording_overclaimed",
            "details": (
                "Governance oversight cannot be said to 'support' alignment. "
                "Reframe as: the bank's source data flags the disclosures as TCFD-aligned and IFRS S2-aligned."
            ),
        })
    # ─────────────────────────────────────────────────────────────────────────

    return {
        "failures": failures,
        "warnings": warnings,
        "failure_count": len(failures),
        "warning_count": len(warnings),
    }


def writer_node(state: GovernanceState) -> GovernanceState:
    is_revision = state["revision_count"] > 0
    feedback = None
    if is_revision:
        judge = state.get("judge_result", {})
        issues = judge.get("required_fixes", [])
        checklist = judge.get("checklist", {})
        false_items = [k for k, v in checklist.items() if v is False and k != "false_count"]
        feedback = (
            "REQUIRED FIXES:\n"
            + "\n".join(f"- {fix}" for fix in issues)
            + "\n\nFAILED CHECKLIST ITEMS:\n"
            + "\n".join(f"- {item}" for item in false_items)
        )
    prompt = build_writer_prompt(state["evidence"], judge_feedback=feedback)
    draft  = call_writer_llm(system_prompt=WRITER_SYSTEM, user_prompt=prompt)
    print(f"\n{'='*50}")
    print(f"WRITER {'(revision ' + str(state['revision_count']) + ')' if is_revision else '(initial draft)'}")
    print(f"Draft length: {len(draft.split())} words")
    print(f"{'='*50}")
    return {**state, "draft": draft.strip(), "status": "judging"}


def judge_node(state: GovernanceState) -> GovernanceState:
    draft = state["draft"]
    deterministic_checks = run_governance_deterministic_checks(draft, state["evidence"])
    judge_prompt = build_judge_prompt(draft, state["evidence"], deterministic_checks=deterministic_checks)
    judge_result = call_judge_llm_json(system_prompt=JUDGE_SYSTEM, user_prompt=judge_prompt)
    judge_result.setdefault("approved", False)
    judge_result.setdefault("approval_status", "approved" if judge_result.get("approved") else "revision_required")
    judge_result.setdefault("main_issues", [])
    judge_result.setdefault("required_fixes", [])
    judge_result.setdefault("checklist", {})
    judge_result["deterministic_prechecks"] = deterministic_checks
    print("\nGOVERNANCE JUDGE RESULT — GPT-5.2 + DATA-AWARE PRECHECKS")
    print(json.dumps(judge_result, indent=2, ensure_ascii=False))
    return {**state, "judge_result": judge_result, "status": "judging"}


GOVERNANCE_REVISER_SYSTEM = """
You are a precise sustainability disclosure reviser.

Revise the existing Governance section using only:
- the supplied compact Governance evidence;
- the availability profile;
- the data-aware deterministic pre-checks; and
- the judge's required fixes.

Rules:
- Fix every judge issue and failed checklist item.
- Preserve correct content that was not criticised.
- Never invent evidence.
- If climate_risk_register is available, describe the risk-register process; do not treat its absence as a boundary.
- Keep the exact six-subsection structure.
- Do not add visible IFRS paragraph references, meeting IDs or internal refs.
- Return only the complete revised Governance section.
""".strip()


def reviser_node(state: GovernanceState) -> GovernanceState:
    if state["revision_count"] >= state["max_revisions"]:
        return {**state, "status": "failed", "final_section": state["draft"]}
    judge = state.get("judge_result", {})
    issues       = judge.get("required_fixes", [])
    checklist    = judge.get("checklist", {})
    deterministic = judge.get("deterministic_prechecks", {})
    false_items  = [k for k, v in checklist.items() if v is False and k != "false_count"]
    revision_prompt = f"""
STRICT GOVERNANCE REQUIREMENTS:
{IFRS_GOVERNANCE_REQUIREMENTS}

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(state["evidence"].get("availability_profile", {}), indent=2, ensure_ascii=False)}

DETERMINISTIC PRE-CHECKS:
{json.dumps(deterministic, indent=2, ensure_ascii=False)}

COMPACT GOVERNANCE EVIDENCE:
{json.dumps(state["evidence"], indent=2, ensure_ascii=False)}

REVISION BOUNDARY:
- Use available evidence when the judge identifies an omission.
- When evidence is unavailable, preserve or improve the accurate boundary statement.
- Never invent a missing process, policy, threshold, trade-off, or assurance scope.
- If climate_risk_register is present, the management responsibility section must describe the risk-register process.

CURRENT DRAFT:
{state["draft"]}

JUDGE REQUIRED FIXES:
{json.dumps(issues, indent=2, ensure_ascii=False)}

FAILED CHECKLIST ITEMS:
{json.dumps(false_items, indent=2, ensure_ascii=False)}

Revise the current draft and return only the complete revised Governance section.
""".strip()
    revised_draft = call_reviser_llm(system_prompt=GOVERNANCE_REVISER_SYSTEM, user_prompt=revision_prompt)
    new_revision_count = state["revision_count"] + 1
    print(f"\nGovernance revised with GPT-4.1 | revision {new_revision_count}")
    return {**state, "draft": revised_draft.strip(), "revision_count": new_revision_count, "status": "judging"}


def finalize_node(state: GovernanceState) -> GovernanceState:
    judge   = state.get("judge_result", {})
    approved = judge.get("approved", False)
    print(f"\n{'='*50}")
    print(f"FINALIZED — {'APPROVED' if approved else 'MAX REVISIONS REACHED'}")
    print(f"  Final score: {judge.get('overall_score')}/10  Revisions: {state['revision_count']}")
    print(f"{'='*50}")
    return {**state, "final_section": state["draft"], "status": "approved" if approved else "failed"}


def route_after_judge(state: GovernanceState) -> str:
    judge         = state.get("judge_result", {})
    approved      = judge.get("approved", False)
    revision_count = state.get("revision_count", 0)
    max_revisions  = state.get("max_revisions", 2)
    if approved:
        return "finalize"
    if revision_count >= max_revisions:
        return "finalize"
    return "revise"


# ============================================================================
# NOTEBOOK CODE CELL 13
# ============================================================================
builder = StateGraph(GovernanceState)
builder.add_node("writer",   writer_node)
builder.add_node("judge",    judge_node)
builder.add_node("reviser",  reviser_node)
builder.add_node("finalize", finalize_node)
builder.add_edge(START,      "writer")
builder.add_edge("writer",   "judge")
builder.add_edge("reviser",  "judge")
builder.add_edge("finalize", END)
builder.add_conditional_edges("judge", route_after_judge, {"revise": "reviser", "finalize": "finalize"})
graph = builder.compile()
print("Governance graph compiled")


# ============================================================================
# NOTEBOOK CODE CELL 14
# ============================================================================
initial_state: GovernanceState = {
    "bank_name":      bank_name,
    "evidence":       evidence,
    "draft":          "",
    "judge_result":   {},
    "revision_count": 0,
    "max_revisions":  2,
    "status":         "drafting",
    "final_section":  "",
    "token_usage":    {},
}
print(f"Starting governance generation for: {bank_name}\n")
result = graph.invoke(initial_state)


# ============================================================================
# NOTEBOOK CODE CELL 15
# ============================================================================
print("\n" + "="*60)
print("FINAL JUDGE RESULT")
print("="*60)
print(json.dumps(result["judge_result"], indent=2, ensure_ascii=False))

print("\n" + "="*60)
print("GOVERNANCE SECTION")
print("="*60)
print(result["final_section"])

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

with open(output_dir / "governance_BANK01.md", "w", encoding="utf-8") as f:
    f.write(result["final_section"])

with open(output_dir / "governance_BANK01_meta.json", "w", encoding="utf-8") as f:
    json.dump({
        "bank_id":        "BANK01",
        "bank_name":      bank_name,
        "section":        "governance",
        "status":          result["status"],
        "approval_status": result["judge_result"].get("approval_status"),
        "final_score":     result["judge_result"].get("overall_score"),
        "revisions":       result["revision_count"],
        "approved":        result["judge_result"].get("approved"),
        "checklist":       result["judge_result"].get("checklist"),
        "issues":          result["judge_result"].get("main_issues"),
        "available_evidence_omitted":     result["judge_result"].get("available_evidence_omitted"),
        "unsupported_claims":             result["judge_result"].get("unsupported_claims"),
        "correctly_disclosed_evidence_boundaries": result["judge_result"].get("correctly_disclosed_evidence_boundaries"),
        "raw_governance_payload_path":    str(RAW_GOVERNANCE_PATH),
        "compact_governance_evidence_path": str(COMPACT_GOVERNANCE_PATH),
    }, f, indent=2, ensure_ascii=False)

print(f"\nSaved: outputs/governance_BANK01.md")


# ============================================================================
# NOTEBOOK CODE CELL 16
# ============================================================================
# ── STRATEGY SECTION GENERATOR ───────────────────────────────

def _safe_float(value, default=0.0):
    try:
        if value is None:
            return default
        if isinstance(value, float) and value != value:
            return default
        return float(value)
    except Exception:
        return default


def _json_clean(obj):
    if isinstance(obj, dict):
        return {k: _json_clean(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_json_clean(v) for v in obj]
    if isinstance(obj, float) and obj != obj:
        return None
    return obj


def _pick_latest(records: list, year: int = 2024) -> dict:
    for r in records:
        if isinstance(r, dict) and _safe_int(r.get("reporting_year")) == year:
            return r
    return records[-1] if records else {}


print("Strategy helper functions ready")


# ============================================================================
# NOTEBOOK CODE CELL 17
# ============================================================================
# ── FAST STRATEGY EVIDENCE EXTRACTOR ────────────────────────
#
# FIX: The strategy payload DOES contain a full targets table (3 records with
# baseline_year, baseline_value, validation_body, milestones, gross_or_net,
# planned_carbon_credits_pct). The writer must use all available target fields.
#
# FIX: business_model_impacts and strategy_tradeoff_decisions are EMPTY lists
# in this payload. The writer must acknowledge this boundary once per subsection.

def _present(v) -> bool:
    if v is None:
        return False
    if isinstance(v, str):
        return v.strip() != "" and v.strip().lower() not in {"nan", "none", "null"}
    try:
        return not (isinstance(v, float) and v != v)
    except Exception:
        return True


def _num(v):
    try:
        if not _present(v):
            return None
        return float(v)
    except Exception:
        return None


def _compact_clean(obj):
    if isinstance(obj, dict):
        cleaned = {k: _compact_clean(v) for k, v in obj.items() if _present(v)}
        return {k: v for k, v in cleaned.items() if v not in ({}, [], None)}
    if isinstance(obj, list):
        cleaned = [_compact_clean(x) for x in obj if _present(x)]
        return [x for x in cleaned if x not in ({}, [], None)]
    if isinstance(obj, float) and obj != obj:
        return None
    return obj


def _top_n(rows: list, key: str, n: int = 5) -> list:
    return sorted(rows, key=lambda r: float(r.get(key) or 0), reverse=True)[:n]


def summarize_strategy_evidence(payload: dict) -> dict:
    metadata      = payload.get("metadata", {})
    bank          = payload.get("bank", {})
    reporting_kpis = payload.get("reporting_kpis", {})

    scenarios     = [s for s in payload.get("climate_scenarios", []) if isinstance(s, dict)]
    risks_all     = [r for r in payload.get("climate_risk_register", []) if isinstance(r, dict)]
    risks_2024    = [r for r in risks_all if r.get("reporting_year") == 2024]
    value_chain   = [v for v in payload.get("value_chain_map", []) if isinstance(v, dict)]
    opportunities = [o for o in payload.get("climate_opportunities", []) if isinstance(o, dict) and o.get("reporting_year") == 2024]

    # ── FIX: use the full targets table directly; it exists in this payload ──
    full_targets  = [t for t in payload.get("targets", []) if isinstance(t, dict)]
    target_summary = reporting_kpis.get("target_summary", [])
    # Prefer full_targets when available (they have baseline, milestones, validation, etc.)
    targets = full_targets if full_targets else target_summary

    physical_risks   = [r for r in risks_2024 if str(r.get("risk_category", "")).startswith("physical")]
    transition_risks = [r for r in risks_2024 if str(r.get("risk_category", "")).startswith("transition")]

    risk_summary = {
        "risk_count_2024":        len(risks_2024),
        "physical_risk_count":    len(physical_risks),
        "transition_risk_count":  len(transition_risks),
        "time_horizons":          sorted({r.get("time_horizon") for r in risks_2024 if _present(r.get("time_horizon"))}),
        "risk_categories":        sorted({r.get("risk_category") for r in risks_2024 if _present(r.get("risk_category"))}),
        "risk_ratings":           sorted({r.get("risk_rating") for r in risks_2024 if _present(r.get("risk_rating"))}),
        "top_risks_by_financial_impact": [
            {
                "risk_id":              r.get("risk_id"),
                "risk_name":            r.get("risk_name"),
                "risk_category":        r.get("risk_category"),
                "risk_rating":          r.get("risk_rating"),
                "time_horizon":         r.get("time_horizon"),
                "financial_impact_meur": r.get("financial_impact_meur"),
                "mitigation_actions":   r.get("mitigation_actions"),
                "scenario_analysis_link": r.get("scenario_analysis_link"),
            }
            for r in _top_n(risks_2024, "financial_impact_meur", 6)
        ],
    }

    scenario_types = sorted({s.get("scenario_type") for s in scenarios if _present(s.get("scenario_type"))})
    scenario_names = sorted({s.get("scenario_name") for s in scenarios if _present(s.get("scenario_name"))})
    frameworks     = sorted({s.get("framework")      for s in scenarios if _present(s.get("framework"))})
    horizon_years  = sorted({s.get("horizon_year")   for s in scenarios if _present(s.get("horizon_year"))})

    def max_record(field: str):
        rows = [s for s in scenarios if _present(s.get(field))]
        if not rows:
            return None
        s = max(rows, key=lambda x: float(x.get(field) or 0))
        return {
            "scenario_id":   s.get("scenario_id"),
            "scenario_name": s.get("scenario_name"),
            "scenario_type": s.get("scenario_type"),
            "horizon":       s.get("horizon"),
            "horizon_year":  s.get("horizon_year"),
            field:           s.get(field),
        }

    resilience_by_type   = {}
    methodology_by_type  = {}
    for s in scenarios:
        stype = s.get("scenario_type")
        if not _present(stype):
            continue
        if _present(s.get("resilience_assessment")):
            resilience_by_type.setdefault(stype, s.get("resilience_assessment"))
        if _present(s.get("methodology_notes")):
            methodology_by_type.setdefault(stype, s.get("methodology_notes"))

    # ── FIX: expose key_assumption_ranges per scenario_type for accurate disclosure ──
    def assumption_range_by_type(field: str):
        by_type = {}
        for s in scenarios:
            stype = s.get("scenario_type")
            val   = s.get(field)
            if not _present(stype) or not _present(val):
                continue
            try:
                fval = float(val)
            except Exception:
                continue
            if stype not in by_type:
                by_type[stype] = {"min": fval, "max": fval}
            else:
                by_type[stype]["min"] = min(by_type[stype]["min"], fval)
                by_type[stype]["max"] = max(by_type[stype]["max"], fval)
        return by_type

    scenario_summary = {
        "available":              bool(scenarios),
        "scenario_count":         len(scenarios),
        "frameworks":             frameworks,
        "scenario_types":         scenario_types,
        "scenario_names":         scenario_names,
        "horizon_years":          horizon_years,
        "methodology_summary_by_scenario_type": methodology_by_type,
        "resilience_assessment_by_scenario_type": resilience_by_type,
        "max_physical_risk_loss_pct_capital":    max_record("physical_risk_loss_pct_capital"),
        "max_transition_risk_loss_pct_capital":  max_record("transition_risk_loss_pct_capital"),
        "max_stranded_assets_estimate_meur":     max_record("stranded_assets_estimate_meur"),
        "max_revenue_at_risk_meur":              max_record("revenue_at_risk_meur"),
        "key_assumption_ranges": {
            "carbon_price_eur_per_tco2e_overall": {
                "min": min(
                    [s.get("carbon_price_assumption_eur_per_tco2e") for s in scenarios
                     if _present(s.get("carbon_price_assumption_eur_per_tco2e"))],
                    default=None,
                ),
                "max": max(
                    [s.get("carbon_price_assumption_eur_per_tco2e") for s in scenarios
                     if _present(s.get("carbon_price_assumption_eur_per_tco2e"))],
                    default=None,
                ),
            },
            "carbon_price_by_scenario_type": assumption_range_by_type("carbon_price_assumption_eur_per_tco2e"),
            "technology_readiness":  sorted({s.get("technology_readiness") for s in scenarios if _present(s.get("technology_readiness"))}),
            "temperature_outcome_c": sorted({s.get("temperature_outcome_c") for s in scenarios if _present(s.get("temperature_outcome_c"))}),
        },
    }

    material_vc   = [v for v in value_chain if v.get("materiality_flag") is True]
    quantified_vc = [v for v in material_vc if _present(v.get("financial_exposure_meur"))]

    value_chain_summary = {
        "available":          bool(value_chain),
        "node_count":         len(value_chain),
        "material_node_count": len(material_vc),
        "qualitative_nodes_without_financial_exposure": len([
            v for v in material_vc if not _present(v.get("financial_exposure_meur"))
        ]),
        "node_types": sorted({v.get("node_type") for v in value_chain if _present(v.get("node_type"))}),
        "largest_quantified_nodes": [
            {
                "node_name":              v.get("node_name"),
                "node_type":              v.get("node_type"),
                "upstream_downstream":    v.get("upstream_downstream"),
                "climate_exposure_type":  v.get("climate_exposure_type"),
                "financial_exposure_meur": v.get("financial_exposure_meur"),
                "climate_risk_description": v.get("climate_risk_description"),
            }
            for v in _top_n(quantified_vc, "financial_exposure_meur", 6)
        ],
        "own_operations_examples": [
            {
                "node_name":             v.get("node_name"),
                "climate_exposure_type": v.get("climate_exposure_type"),
                "scope3_category":       v.get("scope3_category"),
                "climate_risk_description": v.get("climate_risk_description"),
            }
            for v in material_vc
            if v.get("node_type") == "own_operations"
        ][:3],
    }

    opportunity_summary = {
        "available":          bool(opportunities),
        "opportunity_count":  len(opportunities),
        "total_estimated_revenue_impact_meur": round(
            sum(float(o.get("estimated_revenue_impact_meur") or 0) for o in opportunities), 2
        ) if opportunities else None,
        "examples": [
            {
                "opportunity_id":    o.get("opportunity_id"),
                "opportunity_type":  o.get("opportunity_type"),
                "category":          o.get("category"),
                "description":       o.get("description"),
                "estimated_revenue_impact_meur": o.get("estimated_revenue_impact_meur"),
                "time_horizon":      o.get("time_horizon"),
                "confidence_level":  o.get("confidence_level"),
                "linked_risk_category": o.get("linked_risk_category"),
            }
            for o in opportunities
        ],
    }

    financial_2024 = next(
        (f for f in payload.get("financial_summary", []) if isinstance(f, dict) and f.get("reporting_year") == 2024),
        {},
    )

    portfolio_summary = {
        "total_assets_meur":    bank.get("total_assets_meur") or financial_2024.get("total_assets_meur"),
        "total_loans_meur":     bank.get("total_loans_meur") or financial_2024.get("total_loans_meur"),
        "tier1_capital_meur":   financial_2024.get("tier1_capital_meur"),
        "cet1_ratio_pct":       financial_2024.get("cet1_ratio_pct"),
        "total_revenue_meur":   financial_2024.get("total_revenue_meur"),
        "green_loans_meur":     financial_2024.get("green_loans_meur"),
        "green_loans_pct":      financial_2024.get("green_loans_pct") or reporting_kpis.get("green_loans_pct_2024"),
        "financed_emissions_tco2e": reporting_kpis.get("financed_emissions_2024_tco2e"),
        "carbon_intensity_tco2e_per_meur": reporting_kpis.get("carbon_intensity_2024_tco2e_per_meur"),
        # ── FIX: climate capex/opex pulled from financial_2024 (preferred) or kpis ──
        "climate_capex_meur":   financial_2024.get("climate_capex_meur") or reporting_kpis.get("climate_capex_2024_meur"),
        "climate_opex_meur":    financial_2024.get("climate_opex_meur")  or reporting_kpis.get("climate_opex_2024_meur"),
        "high_carbon_sector_exposure_pct":  reporting_kpis.get("high_carbon_sector_exposure_pct"),
        "fossil_fuel_exposure_pct":         reporting_kpis.get("fossil_fuel_exposure_pct"),
        "high_carbon_sector_exposure_meur": reporting_kpis.get("high_carbon_sector_exposure_meur"),
        "fossil_fuel_exposure_meur":        reporting_kpis.get("fossil_fuel_exposure_meur"),
    }

    compact = {
        "bank": {
            "bank_id":           bank.get("bank_id"),
            "bank_name":         bank.get("bank_name"),
            "country":           bank.get("country"),
            "reporting_year":    metadata.get("reporting_year", 2024),
            "reporting_currency": bank.get("reporting_currency"),
            "total_assets_meur": bank.get("total_assets_meur"),
            "total_loans_meur":  bank.get("total_loans_meur"),
        },
        "payload_profile": {
            "source_payload":            "strategy",
            "top_level_keys":            list(payload.keys()),
            "full_targets_table_available": bool(full_targets),
            "target_summary_available":  bool(target_summary),
            "climate_scenarios_count":   len(scenarios),
            "risk_register_count":       len(risks_all),
            "value_chain_node_count":    len(value_chain),
            "opportunity_count":         len(opportunities),
            "business_model_impacts_available": bool(payload.get("business_model_impacts")),
            "strategy_tradeoff_decisions_available": bool(payload.get("strategy_tradeoff_decisions")),
        },
        "risk_summary":        risk_summary,
        "scenario_summary":    scenario_summary,
        "value_chain_summary": value_chain_summary,
        "opportunity_summary": opportunity_summary,
        "portfolio_summary":   portfolio_summary,
        "targets":             targets,
        "business_model_impacts":          payload.get("business_model_impacts", []),
        "strategy_tradeoff_decisions":     payload.get("strategy_tradeoff_decisions", []),
        "evidence_boundaries": {
            "scenario_outputs_are_modelled_estimates": True,
            "opportunity_impacts_are_estimates":       True,
            "do_not_overclaim_resilience":             True,
            "do_not_overclaim_paris_alignment":        True,
            "business_model_detail_available":         bool(payload.get("business_model_impacts")),
            "value_chain_detail_available":            bool(value_chain_summary),
            "tradeoff_evidence_available":             bool(payload.get("strategy_tradeoff_decisions")),
            "high_carbon_and_fossil_exposure_available": (
                _present(portfolio_summary.get("high_carbon_sector_exposure_pct"))
                and _present(portfolio_summary.get("fossil_fuel_exposure_pct"))
            ),
            "full_targets_table_available":  bool(full_targets),
            "target_summary_available":      bool(target_summary),
            # ── FIX: clear target instruction reflecting full table IS available ──
            "target_instruction": (
                "Full target records are available with baseline_year, baseline_value, target_value_pct_reduction, "
                "target_framework, validation_body, gross_or_net, milestones and planned_carbon_credits_pct. "
                "Use all available target fields. Do not treat targets as summary-only."
            ),
            # ── FIX: explicit resource allocation instruction ──────────────────
            "resource_allocation_instruction": (
                "climate_capex_meur and climate_opex_meur are available from the financial summary. "
                "These must be used in the Financial effects and resource allocation subsection."
            ),
            # ── FIX: financed emissions explicit instruction ──────────────────
            "financed_emissions_instruction": (
                "financed_emissions_tco2e and carbon_intensity_tco2e_per_meur are available. "
                "Reference these metrics where financed-emissions scrutiny risk is discussed."
            ),
        },
    }
    return _compact_clean(compact)


strategy_evidence = summarize_strategy_evidence(strategy_payload)

raw_size     = len(json.dumps(strategy_payload, ensure_ascii=False))
compact_size = len(json.dumps(strategy_evidence, ensure_ascii=False))
print(f"Compact Strategy evidence ready. Reduction: {round((1 - compact_size / max(raw_size,1)) * 100, 1)}%")
print(json.dumps({
    "scenario_count":        strategy_evidence["scenario_summary"].get("scenario_count"),
    "risk_count_2024":       strategy_evidence["risk_summary"].get("risk_count_2024"),
    "full_targets_available": strategy_evidence["payload_profile"].get("full_targets_table_available"),
    "business_model_available": strategy_evidence["payload_profile"].get("business_model_impacts_available"),
    "tradeoff_available":    strategy_evidence["payload_profile"].get("strategy_tradeoff_decisions_available"),
    "climate_capex_meur":    strategy_evidence["portfolio_summary"].get("climate_capex_meur"),
    "climate_opex_meur":     strategy_evidence["portfolio_summary"].get("climate_opex_meur"),
    "high_carbon_pct":       strategy_evidence["portfolio_summary"].get("high_carbon_sector_exposure_pct"),
    "fossil_fuel_pct":       strategy_evidence["portfolio_summary"].get("fossil_fuel_exposure_pct"),
}, indent=2, ensure_ascii=False))


# ============================================================================
# NOTEBOOK CODE CELL 18
# ============================================================================
# ── STRATEGY EVIDENCE AVAILABILITY + SAVING ──────────────────

def build_strategy_availability_profile(evidence: dict) -> dict:
    risk         = evidence.get("risk_summary", {})
    scenarios    = evidence.get("scenario_summary", {})
    value_chain  = evidence.get("value_chain_summary", {})
    opportunities = evidence.get("opportunity_summary", {})
    portfolio    = evidence.get("portfolio_summary", {})
    boundaries   = evidence.get("evidence_boundaries", {})
    targets      = evidence.get("targets", [])
    business_model_impacts = evidence.get("business_model_impacts", [])
    tradeoffs    = evidence.get("strategy_tradeoff_decisions", [])

    def present(value) -> bool:
        return value is not None and str(value).strip().lower() not in {"", "none", "null", "nan"}

    scenario_assumptions_available = any(
        present(scenarios.get(field))
        for field in [
            "frameworks", "scenario_types", "scenario_names", "horizon_years",
            "methodology_summary_by_scenario_type", "key_assumption_ranges",
        ]
    )

    quantified_value_chain_available = bool(value_chain.get("largest_quantified_nodes"))
    opportunity_estimates_available  = any(
        present(item.get("estimated_revenue_impact_meur"))
        for item in opportunities.get("examples", [])
        if isinstance(item, dict)
    ) or present(opportunities.get("total_estimated_revenue_impact_meur"))

    return {
        "physical_risk_evidence_available":     bool(risk.get("physical_risk_count")),
        "transition_risk_evidence_available":   bool(risk.get("transition_risk_count")),
        "time_horizons_available":              bool(risk.get("time_horizons")) or bool(scenarios.get("horizon_years")),
        "business_model_detail_available":      bool(business_model_impacts),
        "value_chain_detail_available":         bool(value_chain),
        "quantified_value_chain_exposure_available": quantified_value_chain_available,
        "strategy_tradeoff_evidence_available": bool(tradeoffs),
        "portfolio_financial_metrics_available": any(
            present(portfolio.get(field))
            for field in ["green_loans_meur", "green_loans_pct", "financed_emissions_tco2e", "carbon_intensity_tco2e_per_meur"]
        ),
        # ── FIX: separate resource allocation flag from portfolio metrics ──
        "resource_allocation_evidence_available": (
            present(portfolio.get("climate_capex_meur")) or present(portfolio.get("climate_opex_meur"))
        ),
        "climate_capex_meur":      portfolio.get("climate_capex_meur"),
        "climate_opex_meur":       portfolio.get("climate_opex_meur"),
        "financed_emissions_available": present(portfolio.get("financed_emissions_tco2e")),
        "carbon_intensity_available":   present(portfolio.get("carbon_intensity_tco2e_per_meur")),
        "high_carbon_exposure_available": (
            present(portfolio.get("high_carbon_sector_exposure_pct"))
            or present(portfolio.get("high_carbon_sector_exposure_meur"))
        ),
        "fossil_fuel_exposure_available": (
            present(portfolio.get("fossil_fuel_exposure_pct"))
            or present(portfolio.get("fossil_fuel_exposure_meur"))
        ),
        "climate_opportunities_available":              bool(opportunities.get("examples")),
        "quantified_opportunity_estimates_available":   opportunity_estimates_available,
        "scenario_analysis_available":                  bool(scenarios.get("available")),
        "scenario_assumptions_available":               scenario_assumptions_available,
        "scenario_financial_outputs_available":         any(
            scenarios.get(field) for field in [
                "max_physical_risk_loss_pct_capital", "max_transition_risk_loss_pct_capital",
                "max_stranded_assets_estimate_meur",  "max_revenue_at_risk_meur",
            ]
        ),
        "resilience_evidence_available":    bool(scenarios.get("resilience_assessment_by_scenario_type")),
        "full_targets_table_available":     bool(evidence.get("payload_profile", {}).get("full_targets_table_available")),
        "target_summary_available":         bool(evidence.get("payload_profile", {}).get("target_summary_available")),
        "targets_available":                bool(targets),
        "scenario_outputs_are_modelled_estimates": bool(boundaries.get("scenario_outputs_are_modelled_estimates", True)),
        "opportunity_impacts_are_estimates":       bool(boundaries.get("opportunity_impacts_are_estimates", True)),
        "writer_policy": {
            "use_available_evidence": "Use every material Strategy evidence item that is available and relevant.",
            "handle_unavailable_evidence": (
                "When a material Strategy disclosure is unsupported by available evidence, "
                "state the boundary once in the relevant subsection and do not invent it."
            ),
            "target_boundary": boundaries.get("target_instruction"),
            "resource_allocation_instruction": boundaries.get("resource_allocation_instruction"),
            "financed_emissions_instruction":   boundaries.get("financed_emissions_instruction"),
            "scenario_wording": "Describe scenario financial outputs as modelled estimates, not actual losses.",
            "opportunity_wording": "Describe opportunity revenue impacts as estimates with confidence level, not guaranteed future revenue.",
            "resilience_wording": "Confine resilience statements to the relevant scenario-type assumptions and boundaries.",
            "final_language": "Use 'available evidence', 'available documentation', or 'source data'; do not use 'payload'.",
        },
    }


def add_strategy_traceability(evidence: dict, source_payload_path: Path) -> dict:
    evidence = dict(evidence)
    evidence["availability_profile"] = build_strategy_availability_profile(evidence)
    evidence["source_traceability"] = {
        "source_payload_path": str(source_payload_path),
        "source_tables": [
            "bank", "financial_summary", "climate_scenarios", "climate_risk_register",
            "value_chain_map", "climate_opportunities", "targets", "reporting_kpis",
        ],
        "top_risk_refs": [
            item.get("risk_id") or item.get("risk_name")
            for item in evidence.get("risk_summary", {}).get("top_risks_by_financial_impact", [])
            if item.get("risk_id") or item.get("risk_name")
        ],
        "scenario_refs": [
            item.get("scenario_id")
            for item in [
                evidence.get("scenario_summary", {}).get("max_physical_risk_loss_pct_capital"),
                evidence.get("scenario_summary", {}).get("max_transition_risk_loss_pct_capital"),
                evidence.get("scenario_summary", {}).get("max_stranded_assets_estimate_meur"),
                evidence.get("scenario_summary", {}).get("max_revenue_at_risk_meur"),
            ]
            if isinstance(item, dict) and item.get("scenario_id")
        ],
    }
    return evidence


strategy_evidence = add_strategy_traceability(strategy_evidence, STRATEGY_PAYLOAD_PATH)

raw_strategy_payload = {
    key: strategy_payload.get(key)
    for key in [
        "metadata", "bank", "financial_summary", "climate_scenarios", "climate_risk_register",
        "value_chain_map", "climate_opportunities", "targets", "reporting_kpis",
        "business_model_impacts", "strategy_tradeoff_decisions",
    ]
    if key in strategy_payload
}

RAW_STRATEGY_PATH     = output_dir / "payload_BANK01_strategy_raw.json"
COMPACT_STRATEGY_PATH = output_dir / "compact_strategy_evidence_BANK01.json"

with open(RAW_STRATEGY_PATH, "w", encoding="utf-8") as f:
    json.dump(raw_strategy_payload, f, indent=2, ensure_ascii=False)
with open(COMPACT_STRATEGY_PATH, "w", encoding="utf-8") as f:
    json.dump(strategy_evidence, f, indent=2, ensure_ascii=False)

print("Strategy evidence prepared")
print(f"  resource_allocation_evidence_available: {strategy_evidence['availability_profile']['resource_allocation_evidence_available']}")
print(f"  climate_capex_meur:  {strategy_evidence['availability_profile']['climate_capex_meur']}")
print(f"  climate_opex_meur:   {strategy_evidence['availability_profile']['climate_opex_meur']}")
print(f"  full_targets_table_available: {strategy_evidence['availability_profile']['full_targets_table_available']}")


# ============================================================================
# NOTEBOOK CODE CELL 19
# ============================================================================
class StrategyState(TypedDict):
    bank_name:       str
    evidence:        dict
    draft:           str
    judge_result:    dict
    revision_count:  int
    max_revisions:   int
    status:          Literal["drafting", "judging", "revising", "approved", "failed"]
    final_section:   str
    token_usage:     dict


# ============================================================================
# NOTEBOOK CODE CELL 20
# ============================================================================
# ── STRATEGY REQUIREMENTS ───────────────────────────────────

STRATEGY_REQUIREMENTS = """
STRICT STRATEGY DISCLOSURE REQUIREMENTS FOR THIS SECTION:

Final section title and headings must be exactly:
### Strategy
#### Climate-related risks and opportunities
#### Effects on business model and value chain
#### Effects on strategy and decision-making
#### Financial effects and resource allocation
#### Climate resilience and scenario analysis
#### Strategy limitations and evidence boundaries

Data-aware coverage requirements:
1. Use the Strategy payload as the source of Strategy facts.
2. Use risk register, climate scenarios, value-chain map, climate opportunities, financial summary
   and reporting KPIs when present.
3. Distinguish physical risks from transition risks.
4. Cover time horizons only where risk-register or scenario evidence supports them.
5. Use value-chain evidence when available. If some material nodes lack financial exposure, state that boundary.
6. Opportunity revenue impacts must be described as estimates, not guaranteed revenue.
7. Use high-carbon and fossil-fuel exposure metrics when present.
8. MANDATORY: Use climate capex (EUR 476.95m) and climate opex (EUR 219.32m) in the Financial effects
   and resource allocation subsection. These are directly available in the financial summary.
9. Full target records ARE available; use all available fields including baseline_year, baseline_value,
   target_value_pct_reduction, target_framework, validation_body, gross_or_net, milestones and
   planned_carbon_credits_pct for TGT003. Do not treat targets as summary-only.
10. Scenario financial outputs are modelled estimates, not actual losses.
11. Resilience statements must be scenario-type-specific:
    - orderly: adequate resilience within Pillar 2 thresholds;
    - disorderly: near-term acceptable but medium-term capital consumption material;
    - hot house: physical risk dominant, flood overlay + EPC monitoring as mechanisms.
    Do not blend these into a single general resilience claim.
12. Scenario assumptions must be disclosed: frameworks (NGFS v4), scenario names, horizon years,
    carbon price ranges by scenario type, temperature outcomes, technology readiness levels,
    and key methodology notes.
13. Do not claim overall Paris alignment. Reference paris_alignment flags only at the target level.
14. If business_model_impacts or strategy_tradeoff_decisions are empty, state the evidence boundary
    once; do not invent detailed business-model impacts or trade-off analysis.
15. Reference financed emissions (35,973,167.7 tCO2e) and carbon intensity (1,154.8 tCO2e/MEUR)
    where reputational/financed-emissions scrutiny risk is discussed.
16. Do not include visible IFRS paragraph references or bracketed IFRS tags.
17. Use "available evidence", "available documentation" or "source data" in limitation wording.
    Do not use the word "payload" in the final report.
""".strip()

print("Strategy requirements ready")


# ============================================================================
# NOTEBOOK CODE CELL 21
# ============================================================================
# ── STRATEGY WRITER / JUDGE PROMPTS ─────────────────────────

STRATEGY_WRITER_SYSTEM = """
You are a senior sustainability reporting specialist writing the Strategy section of an
IFRS S1/S2-aligned climate disclosure report for a commercial bank.

Write in a formal, third-person, publication-ready style.

Evidence rules:
- Use only the compact Strategy evidence supplied by the user prompt.
- Do not invent strategy decisions, business-model impacts, trade-offs, value-chain nodes,
  opportunities, target details, scenario assumptions, financial effects or resilience conclusions.
- If evidence is missing, state the limitation once in report-style language.
- Do not use the word "payload" in the final section.
- Do not include IFRS paragraph references or bracketed evidence tags.
- Do not use markdown tables.

Output rules:
- Return only the complete Strategy section.
- Keep exactly the six required subsections.
""".strip()


def build_strategy_writer_prompt(evidence: dict, judge_feedback=None) -> str:
    profile   = evidence.get("availability_profile", {})
    boundaries = evidence.get("evidence_boundaries", {})
    portfolio  = evidence.get("portfolio_summary", {})
    instructions = []

    if profile.get("physical_risk_evidence_available") and profile.get("transition_risk_evidence_available"):
        instructions.append("- Distinguish and describe both physical and transition risks using the supplied risk examples.")
    else:
        instructions.append("- Describe only the risk types evidenced; do not invent missing risk categories.")

    if profile.get("time_horizons_available"):
        instructions.append("- Use available short-, medium- and long-term horizons where supported by risk/scenario evidence.")

    if profile.get("business_model_detail_available"):
        instructions.append("- Explain supplied business-model impacts and transmission channels.")
    else:
        instructions.append(
            "- business_model_impacts is empty in the source data. State once in 'Effects on business model and value chain' "
            "that detailed stand-alone business-model impact analysis is not available, then use lending/portfolio/"
            "value-chain and financial evidence to describe identifiable transmission channels."
        )

    if profile.get("value_chain_detail_available"):
        instructions.append("- Use the value-chain map, including own operations, suppliers and financing counterparties.")
        if profile.get("quantified_value_chain_exposure_available"):
            instructions.append("- Include quantified value-chain exposures and note that some material nodes are qualitative where financial exposure is absent.")

    if profile.get("strategy_tradeoff_evidence_available"):
        instructions.append("- Describe only documented Strategy trade-offs provided in evidence.")
    else:
        instructions.append(
            "- strategy_tradeoff_decisions is empty. State once in 'Effects on strategy and decision-making' "
            "that no quantified or documented trade-off analysis is available in the source data."
        )

    # ── FIX: mandatory resource allocation instruction ──────────────────────
    if profile.get("resource_allocation_evidence_available"):
        capex = profile.get("climate_capex_meur")
        opex  = profile.get("climate_opex_meur")
        instructions.append(
            f"- MANDATORY: In 'Financial effects and resource allocation', include climate capex of "
            f"EUR {capex}m and climate opex of EUR {opex}m. These are directly evidenced in the "
            "financial summary and must not be omitted."
        )
    # ─────────────────────────────────────────────────────────────────────────

    if profile.get("high_carbon_exposure_available"):
        instructions.append("- Use high-carbon sector exposure metrics.")
    if profile.get("fossil_fuel_exposure_available"):
        instructions.append("- Use fossil-fuel exposure metrics.")

    # ── FIX: financed emissions instruction ─────────────────────────────────
    if profile.get("financed_emissions_available"):
        instructions.append(
            f"- Reference financed emissions ({portfolio.get('financed_emissions_tco2e'):,.1f} tCO2e) and "
            f"carbon intensity ({portfolio.get('carbon_intensity_tco2e_per_meur')} tCO2e/MEUR) "
            "where financed-emissions scrutiny risk is discussed."
        )
    # ─────────────────────────────────────────────────────────────────────────

    if profile.get("climate_opportunities_available"):
        instructions.append("- Describe the concrete climate opportunities supplied in evidence.")
        if profile.get("quantified_opportunity_estimates_available"):
            instructions.append("- Use opportunity revenue estimates cautiously and identify them as estimates with confidence levels.")

    # ── FIX: scenario assumptions are mandatory ─────────────────────────────
    if profile.get("scenario_analysis_available"):
        instructions.append(
            "- In 'Climate resilience and scenario analysis', disclose: (a) scenario families "
            "(NGFS v4 Net Zero 2050/Below 2°C orderly; Divergent Net Zero/Delayed Transition disorderly; "
            "NDC/Current Policies hot house); (b) horizon years (2025, 2030, 2050); "
            "(c) carbon price range by scenario type; (d) temperature outcomes (1.5°C, 1.8°C, 2.5°C, 3.0°C); "
            "(e) technology readiness (high/medium/low per scenario type); "
            "(f) key methodology notes (top-down sector-pathway approach, NGFS IAM outputs, "
            "RCP 8.5 physical risk for 2050 horizon)."
        )
    # ─────────────────────────────────────────────────────────────────────────

    # ── FIX: scenario-type-specific resilience instructions ─────────────────
    if profile.get("resilience_evidence_available"):
        instructions.append(
            "- Present resilience conclusions per scenario type separately:\n"
            "  • orderly: adequate resilience, transition losses within Pillar 2 thresholds;\n"
            "  • disorderly: near-term acceptable, medium-term capital consumption from stranded asset "
            "impairments is material, sector exposure limits and glide-path monitoring are primary mechanisms, "
            "additional buffers may be required post-2030;\n"
            "  • hot house: physical risk dominant, flood-zone overlay and EPC-based collateral monitoring "
            "are the deployed resilience mechanisms.\n"
            "  Do NOT produce a single general resilience conclusion. Do NOT say the bank is 'generally resilient'."
        )
    # ─────────────────────────────────────────────────────────────────────────

    # ── FIX: full targets instruction ────────────────────────────────────────
    if profile.get("full_targets_table_available"):
        instructions.append(
            "- Full target records are available. For each target use: target_type, scope, metric, "
            "baseline_year, baseline_value, target_year, target_value_pct_reduction, target_framework, "
            "status, gross_or_net, validation_body, sectoral_decarbonisation flag, interim milestones "
            "where present, and planned_carbon_credits_pct/type where applicable (TGT003 has 8.2% "
            "technology removal credits planned)."
        )
    elif profile.get("target_summary_available"):
        instructions.append(
            "- Only target_summary is available. Use target type, scope, status, year, framework and "
            "progress only; do not invent baselines, validation bodies, milestones, gross/net status or planned credits."
        )
    # ─────────────────────────────────────────────────────────────────────────

    feedback_block = ""
    if judge_feedback:
        feedback_block = (
            "JUDGE FEEDBACK TO ADDRESS:\n"
            f"{judge_feedback}\n\n"
            "REVISION POLICY:\n"
            "- Fix omissions using available evidence.\n"
            "- Preserve accurate evidence-boundary statements where evidence is unavailable.\n"
            "- Never invent evidence to satisfy the judge."
        )

    return f"""
{STRATEGY_REQUIREMENTS}

BANK:
{evidence['bank']['bank_name']} ({evidence['bank']['bank_id']})

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(profile, indent=2, ensure_ascii=False)}

DATA-AWARE WRITING INSTRUCTIONS:
{chr(10).join(instructions)}

COMPACT STRATEGY EVIDENCE — USE ONLY THIS DATA:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

IMPORTANT DATA BOUNDARIES:
- Strategy payload tables available: {evidence.get('payload_profile', {}).get('top_level_keys')}
- Full targets table available: {evidence.get('payload_profile', {}).get('full_targets_table_available')}
- Business model impacts available: {evidence.get('payload_profile', {}).get('business_model_impacts_available')}
- Strategy tradeoff decisions available: {evidence.get('payload_profile', {}).get('strategy_tradeoff_decisions_available')}
- Target instruction: {boundaries.get('target_instruction')}
- Resource allocation instruction: {boundaries.get('resource_allocation_instruction')}
- Financed emissions instruction: {boundaries.get('financed_emissions_instruction')}
- Scenario outputs are modelled estimates: {boundaries.get('scenario_outputs_are_modelled_estimates')}
- Opportunity impacts are estimates: {boundaries.get('opportunity_impacts_are_estimates')}

GENERAL WRITING RULES:
- Do not claim that opportunities are guaranteed revenue.
- Do not claim that scenario losses are actual losses.
- Do not produce a single general bank resilience claim; keep resilience scenario-type-specific.
- Do not claim overall Paris alignment.
- Use exact figures from compact evidence.
- Forward-looking statements (e.g., expected green loan growth) must be anchored to documented
  evidence, management intent or target commitments — not inferred.
- Mention evidence boundaries only once per subsection.

{feedback_block}

Write the complete Strategy section only.
""".strip()


STRATEGY_JUDGE_SYSTEM = """
You are a strict sustainability-reporting judge for an IFRS S1/S2-aligned bank climate Strategy disclosure.

You evaluate whether the Strategy section is:
- supported by compact evidence;
- aligned with the Strategy disclosure checklist;
- transparent about missing evidence;
- free from unsupported claims about targets, opportunities, scenarios, business model,
  value chain, Paris alignment and resilience.

Return valid JSON only.
""".strip()


def build_strategy_judge_prompt(draft: str, evidence: dict, deterministic_checks: dict = None) -> str:
    profile = evidence.get("availability_profile", {})
    deterministic_checks = deterministic_checks or {}

    return f"""
Evaluate the Strategy draft against the compact evidence, availability profile and deterministic pre-checks.

DRAFT:
{draft}

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(profile, indent=2, ensure_ascii=False)}

DETERMINISTIC PRE-CHECKS:
{json.dumps(deterministic_checks, indent=2, ensure_ascii=False)}

COMPACT STRATEGY EVIDENCE:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

EVALUATION PRINCIPLES:
1. Penalise claims that contradict evidence, invent unsupported information, or overstate estimates.
2. Penalise omission when material evidence is available but not used.
3. Do not demand unavailable evidence. Correctly disclosed boundaries are acceptable but lower completeness.
4. Verify use of physical and transition risks, time horizons, scenario evidence, value-chain map,
   opportunities, portfolio metrics and resource allocation.
5. Full target records ARE available. Verify that baseline_year, baseline_value, target_value_pct_reduction,
   target_framework, validation_body, gross_or_net, milestones and planned_carbon_credits_pct (for TGT003) are used.
6. Climate capex (EUR 476.95m) and climate opex (EUR 219.32m) are available. Penalise omission.
7. Financed emissions (35,973,167.7 tCO2e) and carbon intensity (1,154.8 tCO2e/MEUR) are available.
   Penalise if not referenced where financed-emissions scrutiny risk is discussed.
8. Opportunity revenue impacts must be described as estimates, not guaranteed future revenue.
9. Scenario financial outputs must be described as modelled estimates.
10. Resilience claims must be scenario-type-specific (orderly / disorderly / hot house).
    A single general resilience conclusion is not acceptable.
11. Scenario assumptions must be disclosed: framework (NGFS v4), scenario names, horizon years,
    carbon price ranges, temperature outcomes and technology readiness.
12. If business_model_impacts are empty, the draft must not claim detailed business-model impact analysis.
13. If strategy_tradeoff_decisions are empty, the draft must not invent trade-off analysis.
14. Overall Paris-alignment claims are not allowed unless directly supported and carefully bounded.
15. No visible IFRS paragraph references or bracketed tags.

SCORING:
- 9–10: materially complete, directly evidenced, no material boundary required.
- 8: strong with one material boundary or minor evidence-use issue.
- 7: usable with multiple correctly disclosed boundaries.
- 6: revision required because available evidence was omitted, contradicted or overstated.
- 5 or below: major factual, evidence or overclaiming failures.

Return valid JSON only. Keep arrays concise:
{{
  "overall_score": <integer 1-10>,
  "evidence_support_score": <integer 1-10>,
  "ifrs_alignment_score": <integer 1-10>,
  "specificity_score": <integer 1-10>,
  "hallucination_risk": "<low|medium|high>",
  "approval_status": "<approved|approved_with_limitations|revision_required|rejected>",
  "approved": <true if approved or approved_with_limitations, otherwise false>,
  "checklist": {{
    "required_structure_present": <true/false>,
    "physical_and_transition_risks_used_correctly": <true/false>,
    "time_horizons_used_correctly": <true/false>,
    "business_model_handled_according_to_availability": <true/false>,
    "value_chain_used_correctly": <true/false>,
    "strategy_tradeoffs_handled_according_to_availability": <true/false>,
    "financial_effects_used_correctly": <true/false>,
    "resource_allocation_used_correctly": <true/false>,
    "climate_capex_and_opex_included": <true/false>,
    "high_carbon_and_fossil_exposure_used_if_available": <true/false>,
    "financed_emissions_referenced_where_relevant": <true/false>,
    "opportunities_used_correctly": <true/false>,
    "targets_handled_according_to_availability": <true/false>,
    "full_target_fields_used_including_baseline_milestones_validation": <true/false>,
    "scenario_analysis_used_correctly": <true/false>,
    "scenario_assumptions_disclosed": <true/false>,
    "resilience_scenario_type_specific": <true/false>,
    "resilience_not_overclaimed": <true/false>,
    "paris_alignment_not_overclaimed": <true/false>,
    "modelled_estimates_not_overstated": <true/false>,
    "no_visible_ifrs_refs": <true/false>,
    "no_unsupported_strong_claims": <true/false>
  }},
  "available_evidence_omitted": [<specific available evidence omitted>],
  "unsupported_claims": [<specific unsupported claims>],
  "correctly_disclosed_evidence_boundaries": [<accurate boundary statements>],
  "main_issues": [<specific issues>],
  "required_fixes": [<actionable evidence-aware fixes>]
}}
""".strip()


print("Strategy prompts ready")


# ============================================================================
# NOTEBOOK CODE CELL 23
# ============================================================================
# ── STRATEGY LANGGRAPH NODES ────────────────────────────────

def run_strategy_deterministic_checks(draft: str, evidence: dict) -> dict:
    text    = draft or ""
    text_l  = text.lower()
    profile = evidence.get("availability_profile", {})

    required_headings = [
        "#### Climate-related risks and opportunities",
        "#### Effects on business model and value chain",
        "#### Effects on strategy and decision-making",
        "#### Financial effects and resource allocation",
        "#### Climate resilience and scenario analysis",
        "#### Strategy limitations and evidence boundaries",
    ]
    missing_headings = [h for h in required_headings if h not in text]

    warnings = []
    failures = []

    if missing_headings:
        failures.append({"check": "missing_required_headings", "details": missing_headings})

    if re.search(r"IFRS\s*S?[12]?\s*§|§\s*\d|\[IFRS", text):
        failures.append({"check": "visible_ifrs_references", "details": "Visible IFRS references/tags found."})

    unsupported_strong_phrases = [
        "fully resilient", "guarantees", "guaranteed",
        "proves resilience", "fully paris aligned", "overall paris aligned", "no material risk",
    ]
    found_strong = [p for p in unsupported_strong_phrases if p in text_l]
    if found_strong:
        failures.append({"check": "unsupported_strong_strategy_language", "details": found_strong})

    # ── FIX: capex/opex are mandatory ───────────────────────────────────────
    if profile.get("resource_allocation_evidence_available"):
        capex_str = str(profile.get("climate_capex_meur") or "476.95")
        opex_str  = str(profile.get("climate_opex_meur")  or "219.32")
        capex_val = capex_str.replace(".0", "").split(".")[0]
        opex_val  = opex_str.replace(".0", "").split(".")[0]
        if capex_val not in text and "476" not in text:
            failures.append({
                "check": "climate_capex_omitted",
                "details": f"Climate capex EUR {capex_str}m is available and must appear in Financial effects subsection.",
            })
        if opex_val not in text and "219" not in text:
            failures.append({
                "check": "climate_opex_omitted",
                "details": f"Climate opex EUR {opex_str}m is available and must appear in Financial effects subsection.",
            })
    # ─────────────────────────────────────────────────────────────────────────

    # ── FIX: scenario assumptions disclosure check ───────────────────────────
    if profile.get("scenario_analysis_available"):
        assumption_terms = ["carbon price", "temperature", "technology readiness", "ngfs", "methodology"]
        if not any(term in text_l for term in assumption_terms):
            failures.append({
                "check": "scenario_assumptions_not_disclosed",
                "details": "Scenario assumptions (carbon price, temperature outcomes, methodology) are available but not disclosed.",
            })
    # ─────────────────────────────────────────────────────────────────────────

    # ── FIX: resilience must be scenario-type-specific ───────────────────────
    if profile.get("resilience_evidence_available"):
        resilience_types_mentioned = sum([
            1 for term in ["orderly", "disorderly", "hot house"]
            if term in text_l
        ])
        if resilience_types_mentioned < 2:
            failures.append({
                "check": "resilience_not_scenario_type_specific",
                "details": (
                    "Resilience evidence covers orderly, disorderly and hot house scenarios. "
                    "The draft must present scenario-type-specific resilience conclusions, not a single general claim."
                ),
            })
    # ─────────────────────────────────────────────────────────────────────────

    if profile.get("climate_opportunities_available") and "opportun" not in text_l:
        warnings.append({"check": "opportunities_may_be_omitted", "details": "Climate opportunity evidence exists but opportunity wording is not detected."})

    if profile.get("quantified_opportunity_estimates_available"):
        if "estimate" not in text_l and "estimated" not in text_l:
            failures.append({"check": "opportunity_estimates_not_labelled", "details": "Opportunity revenue impacts should be described as estimates."})

    if profile.get("scenario_financial_outputs_available"):
        estimate_terms = ["modelled", "modeled", "scenario", "estimate", "projected"]
        if not any(term in text_l for term in estimate_terms):
            failures.append({"check": "scenario_outputs_not_labelled_as_estimates", "details": "Scenario financial outputs should be labelled as modelled estimates."})

    if profile.get("value_chain_detail_available") and "value chain" not in text_l:
        warnings.append({"check": "value_chain_may_be_omitted", "details": "Value-chain evidence exists but value-chain wording is not detected."})

    if profile.get("high_carbon_exposure_available") and "high-carbon" not in text_l and "high carbon" not in text_l:
        warnings.append({"check": "high_carbon_exposure_may_be_omitted", "details": "High-carbon exposure metrics exist but may be omitted."})

    if profile.get("fossil_fuel_exposure_available") and "fossil" not in text_l:
        warnings.append({"check": "fossil_fuel_exposure_may_be_omitted", "details": "Fossil-fuel exposure metrics exist but may be omitted."})

    # ── FIX: full targets check — warn if target baselines not used ──────────
    if profile.get("full_targets_table_available"):
        if "baseline" not in text_l:
            warnings.append({
                "check": "full_target_baseline_may_be_omitted",
                "details": "Full target records include baseline_year and baseline_value which should be disclosed.",
            })
        if "validation" not in text_l and "sbti" not in text_l and "unep" not in text_l:
            warnings.append({
                "check": "target_validation_body_may_be_omitted",
                "details": "Full target records include validation_body (SBTi, UNEP FI) which should be disclosed.",
            })
    # ─────────────────────────────────────────────────────────────────────────

    if not profile.get("strategy_tradeoff_evidence_available"):
        if ("trade-off" in text_l or "tradeoff" in text_l) and \
           not any(term in text_l for term in ["not available", "not documented", "does not provide", "no quantified"]):
            failures.append({"check": "strategy_tradeoff_overclaimed", "details": "Trade-off language appears without available trade-off evidence or limitation."})

    if not profile.get("business_model_detail_available"):
        if "detailed business model" in text_l and \
           not any(term in text_l for term in ["limited", "not available", "not detailed"]):
            failures.append({"check": "business_model_detail_overclaimed", "details": "Detailed business-model impact evidence is absent."})

    return {
        "failures": failures,
        "warnings": warnings,
        "failure_count": len(failures),
        "warning_count": len(warnings),
    }


def strategy_writer_node(state: StrategyState) -> StrategyState:
    is_revision = state["revision_count"] > 0
    feedback = None
    if is_revision:
        judge = state.get("judge_result", {})
        issues = judge.get("required_fixes", [])
        checklist = judge.get("checklist", {})
        false_items = [k for k, v in checklist.items() if v is False and k != "false_count"]
        feedback = (
            "REQUIRED FIXES:\n"
            + "\n".join(f"- {fix}" for fix in issues)
            + "\n\nFAILED CHECKLIST ITEMS:\n"
            + "\n".join(f"- {item}" for item in false_items)
        )
    prompt = build_strategy_writer_prompt(state["evidence"], judge_feedback=feedback)
    draft  = call_writer_llm(system_prompt=STRATEGY_WRITER_SYSTEM, user_prompt=prompt)
    print(f"\n{'='*50}")
    print(f"STRATEGY WRITER {'(revision ' + str(state['revision_count']) + ')' if is_revision else '(initial draft)'}")
    print(f"Draft length: {len(draft.split())} words")
    print(f"{'='*50}")
    return {**state, "draft": draft.strip(), "status": "judging"}


def strategy_judge_node(state: StrategyState) -> StrategyState:
    draft = state["draft"]
    deterministic_checks = run_strategy_deterministic_checks(draft, state["evidence"])
    prompt = build_strategy_judge_prompt(draft, state["evidence"], deterministic_checks=deterministic_checks)
    judge_result = call_judge_llm_json(system_prompt=STRATEGY_JUDGE_SYSTEM, user_prompt=prompt)
    judge_result.setdefault("approved", False)
    judge_result.setdefault("approval_status", "approved" if judge_result.get("approved") else "revision_required")
    judge_result.setdefault("main_issues", [])
    judge_result.setdefault("required_fixes", [])
    judge_result.setdefault("checklist", {})
    judge_result["deterministic_prechecks"] = deterministic_checks
    print("\nSTRATEGY JUDGE RESULT — GPT-5.2 + DATA-AWARE PRECHECKS")
    print(json.dumps(judge_result, indent=2, ensure_ascii=False))
    approved = bool(judge_result.get("approved"))
    return {**state, "judge_result": judge_result, "status": "approved" if approved else "revising"}


STRATEGY_REVISER_SYSTEM = """
You are a precise sustainability disclosure reviser.

Revise the existing Strategy section using only:
- the supplied compact Strategy evidence;
- the availability profile;
- the data-aware deterministic pre-checks; and
- the judge's required fixes.

Rules:
- Fix every judge issue and failed checklist item.
- Preserve correct content that was not criticised.
- Never invent evidence.
- Climate capex and opex are available and must appear in the Financial effects subsection.
- Full target records are available; use baseline_year, baseline_value, target_value_pct_reduction,
  validation_body, gross_or_net, milestones and planned_carbon_credits_pct where available.
- Resilience must be scenario-type-specific (orderly / disorderly / hot house), not a general claim.
- Scenario assumptions (NGFS v4 framework, carbon price ranges, temperature outcomes,
  technology readiness, methodology) must be disclosed.
- Clearly distinguish estimates, scenarios and actual financial effects.
- Keep the exact six-subsection structure.
- Do not add visible IFRS paragraph references.
- Return only the complete revised Strategy section.
""".strip()


def strategy_reviser_node(state: StrategyState) -> StrategyState:
    if state["revision_count"] >= state["max_revisions"]:
        return {**state, "status": "failed", "final_section": state["draft"]}
    judge = state.get("judge_result", {})
    issues       = judge.get("required_fixes", [])
    checklist    = judge.get("checklist", {})
    deterministic = judge.get("deterministic_prechecks", {})
    false_items  = [k for k, v in checklist.items() if v is False and k != "false_count"]
    revision_prompt = f"""
STRICT STRATEGY REQUIREMENTS:
{STRATEGY_REQUIREMENTS}

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(state["evidence"].get("availability_profile", {}), indent=2, ensure_ascii=False)}

DETERMINISTIC PRE-CHECKS:
{json.dumps(deterministic, indent=2, ensure_ascii=False)}

COMPACT STRATEGY EVIDENCE:
{json.dumps(state["evidence"], indent=2, ensure_ascii=False)}

REVISION BOUNDARY:
- Use available evidence when the judge identifies an omission.
- Preserve or improve accurate boundary statements where evidence is unavailable.
- Never invent missing opportunities, trade-offs, business-model impacts, value-chain effects,
  scenario assumptions, financial impacts, or resilience conclusions beyond available evidence.

CURRENT DRAFT:
{state["draft"]}

JUDGE REQUIRED FIXES:
{json.dumps(issues, indent=2, ensure_ascii=False)}

FAILED CHECKLIST ITEMS:
{json.dumps(false_items, indent=2, ensure_ascii=False)}

Revise the current draft and return only the complete revised Strategy section.
""".strip()
    revised_draft = call_reviser_llm(system_prompt=STRATEGY_REVISER_SYSTEM, user_prompt=revision_prompt)
    new_revision_count = state["revision_count"] + 1
    print(f"\nStrategy revised with GPT-4.1 | revision {new_revision_count}")
    return {**state, "draft": revised_draft.strip(), "revision_count": new_revision_count, "status": "judging"}


def strategy_finalize_node(state: StrategyState) -> StrategyState:
    judge   = state.get("judge_result", {})
    approved = judge.get("approved", False)
    print(f"\n{'='*50}")
    print(f"FINALIZED STRATEGY — {'APPROVED' if approved else 'MAX REVISIONS REACHED'}")
    print(f"  Final score: {judge.get('overall_score')}/10  Revisions: {state['revision_count']}")
    print(f"{'='*50}")
    return {**state, "final_section": state["draft"], "status": "approved" if approved else "failed"}


def strategy_should_continue(state: StrategyState) -> str:
    judge = state.get("judge_result", {})
    if judge.get("approved", False):
        return "finalize"
    if state.get("revision_count", 0) >= state.get("max_revisions", 1):
        return "finalize"
    return "reviser"


# ============================================================================
# NOTEBOOK CODE CELL 24
# ============================================================================
strategy_builder = StateGraph(StrategyState)
strategy_builder.add_node("writer",   strategy_writer_node)
strategy_builder.add_node("judge",    strategy_judge_node)
strategy_builder.add_node("reviser",  strategy_reviser_node)
strategy_builder.add_node("finalize", strategy_finalize_node)
strategy_builder.add_edge(START, "writer")
strategy_builder.add_edge("writer", "judge")
strategy_builder.add_conditional_edges(
    "judge", strategy_should_continue,
    {"finalize": "finalize", "reviser": "reviser"}
)
strategy_builder.add_edge("reviser", "judge")
strategy_builder.add_edge("finalize", END)
strategy_graph = strategy_builder.compile()
print("Strategy graph compiled")


# ============================================================================
# NOTEBOOK CODE CELL 25
# ============================================================================
strategy_initial_state: StrategyState = {
    "bank_name":      bank_name,
    "evidence":       strategy_evidence,
    "draft":          "",
    "judge_result":   {},
    "revision_count": 0,
    "max_revisions":  1,
    "status":         "drafting",
    "final_section":  "",
    "token_usage":    {},
}
print(f"Starting strategy generation for: {bank_name}\n")
strategy_result = strategy_graph.invoke(strategy_initial_state)


# ============================================================================
# NOTEBOOK CODE CELL 26
# ============================================================================
print("\n" + "="*60)
print("FINAL STRATEGY JUDGE RESULT")
print("="*60)
print(json.dumps(strategy_result["judge_result"], indent=2, ensure_ascii=False))

print("\n" + "="*60)
print("STRATEGY SECTION")
print("="*60)
print(strategy_result["final_section"])

with open(output_dir / "strategy_BANK01.md", "w", encoding="utf-8") as f:
    f.write(strategy_result["final_section"])

with open(output_dir / "strategy_BANK01_meta.json", "w", encoding="utf-8") as f:
    json.dump({
        "bank_id":        "BANK01",
        "bank_name":      bank_name,
        "section":        "strategy",
        "status":         strategy_result["status"],
        "approval_status": strategy_result["judge_result"].get("approval_status"),
        "final_score":    strategy_result["judge_result"].get("overall_score"),
        "revisions":      strategy_result["revision_count"],
        "approved":       strategy_result["judge_result"].get("approved"),
        "checklist":      strategy_result["judge_result"].get("checklist"),
        "issues":         strategy_result["judge_result"].get("main_issues"),
        "available_evidence_omitted":     strategy_result["judge_result"].get("available_evidence_omitted"),
        "unsupported_claims":             strategy_result["judge_result"].get("unsupported_claims"),
        "correctly_disclosed_evidence_boundaries": strategy_result["judge_result"].get("correctly_disclosed_evidence_boundaries"),
        "raw_strategy_payload_path":      str(RAW_STRATEGY_PATH),
        "compact_strategy_evidence_path": str(COMPACT_STRATEGY_PATH),
    }, f, indent=2, ensure_ascii=False)

print("\nSaved: outputs/strategy_BANK01.md")

Loaded .env from: c:\Users\BV426BP\Documents\IFRS Data\.env
Role-based Azure OpenAI configuration loaded
Role-specific LLM helper functions ready
Loaded Governance payload for: Eurolux Universal Bank AG
Governance top-level keys: ['metadata', 'bank', 'governance', 'board_minutes', 'climate_risk_register', 'reporting_kpis']
Strategy top-level keys:   ['metadata', 'bank', 'financial_summary', 'climate_scenarios', 'climate_risk_register', 'value_chain_map', 'climate_opportunities', 'targets', 'reporting_kpis']
Evidence extracted for: Eurolux Universal Bank AG
Governance payload risk register present: True
Board decisions selected: 6
Management evidence risk register available: True
Management risk count 2024: 8
Committee decision summary:
  full_board: decisions=['Approved 2024 ESG report for publication', 'Approved carbon credit procurement budget', 'Endorsed updated transition plan', 'Approved climate scenario analysis methodology'], topics=['carbon_credit_approval', 'exec_remuneration_